# Thực hành Chương 2 — Thu thập dữ liệu lớn

**Học phần:** Xử lý dữ liệu lớn (MTDL522) — Chương trình Thạc sĩ Khoa học máy tính

**Môi trường.**
- *Google Colab:* mở notebook, chạy lần lượt từ ô đầu tiên; ô cài đặt tự cài `pyspark==3.5.3` nếu chưa có (Colab đã có Java).
- *Máy cá nhân:* Python 3.9–3.12, Java 8/11/17, và `pip install pyspark==3.5.3 pandas pyarrow matplotlib scikit-learn`.
- Toàn bộ dữ liệu được **sinh bằng mã (mô phỏng, có `seed`)** và ghi vào thư mục tương đối `./data_c2/`; không cần tệp ngoài. Các con số trong notebook **không phải số liệu thực tế** của bất kỳ trường hay hệ thống nào.

**Quy ước.** Ô `# BÀI TẬP …` là chỗ học viên điền mã (thay `...`/`None`). Ngay sau mỗi bài tập là **ô tự kiểm tra**; khi đúng, ô in `✓ Đạt`. Cuối mỗi phần có **câu hỏi thảo luận** (trả lời ngắn bằng markdown trong notebook của mình).

## 1. Cài đặt và khởi tạo môi trường

Ô dưới đây: (i) phát hiện Colab và cài PySpark nếu thiếu; (ii) kiểm tra Java; (iii) tạo `SparkSession` cục bộ `local[*]`.
Múi giờ của trình điều khiển Python và của Spark được đặt cùng là `Asia/Ho_Chi_Minh` để các mốc thời gian (watermark, thời điểm sự kiện) không bị lệch khi chạy trên Colab (mặc định UTC).

In [ ]:
import os, sys, re, json, time, shutil, subprocess, sqlite3, random, math, zlib, unicodedata
from datetime import datetime, date, timedelta
from functools import reduce

# Đồng bộ múi giờ Python ↔ JVM (đặt trước khi khởi động Spark)
os.environ["TZ"] = "Asia/Ho_Chi_Minh"
if hasattr(time, "tzset"):
    time.tzset()
# Bảo đảm tiến trình worker của Spark dùng cùng trình thông dịch Python với notebook
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

IN_COLAB = "google.colab" in sys.modules
try:
    import pyspark
except ImportError:
    %pip install -q pyspark==3.5.3
    import pyspark
print("Môi trường:", "Google Colab" if IN_COLAB else "máy cá nhân", "| Python", sys.version.split()[0], "| PySpark", pyspark.__version__)


def kiem_tra_java():
    """Tìm Java và kiểm tra phiên bản (PySpark 3.5 hỗ trợ Java 8, 11, 17)."""
    java = None
    if os.environ.get("JAVA_HOME"):
        cand = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
        java = cand if os.path.exists(cand) else None
    java = java or shutil.which("java")
    if java is None:
        print("⚠ Không tìm thấy Java. Cài một trong các bản Java 8/11/17 rồi khởi động lại kernel:\n"
              "  - Colab/Ubuntu: !apt-get install -y openjdk-17-jdk-headless\n"
              "  - macOS (Homebrew): brew install openjdk@17\n"
              "  - Windows: cài Eclipse Temurin 17, đặt biến môi trường JAVA_HOME")
        return None
    out = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
    m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
    major = int(m.group(1)) if m else None
    if major == 1 and m.group(2):
        major = int(m.group(2))
    print(f"Java: {out.splitlines()[0] if out else java}")
    if major not in (8, 11, 17):
        print(f"⚠ Java {major} chưa được PySpark 3.5 hỗ trợ chính thức; nên dùng Java 8/11/17.")
    return major


JAVA_MAJOR = kiem_tra_java()

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F, types as T

spark = (SparkSession.builder
         .master("local[*]")
         .appName("TH_C2_ThuThapDuLieuLon")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse-c2"))
         .config("spark.sql.session.timeZone", "Asia/Ho_Chi_Minh")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.driver.memory", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master =", spark.sparkContext.master)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12, "axes.titlesize": 13, "figure.dpi": 100})
pd.set_option("display.width", 140)


def dat(ten):
    """In thông báo khi một ô tự kiểm tra thành công."""
    print(f"✓ Đạt — {ten}")

## 2. Sinh dữ liệu mô phỏng cho trung tâm dữ liệu của một trường đại học

Bối cảnh: trung tâm dữ liệu của trường cần thu nhận dữ liệu từ ba nguồn có đặc trưng khác nhau (xem slide *Pipeline thu thập cho trung tâm dữ liệu đại học*):

| Nguồn | Hình thức mô phỏng | Đặc trưng |
|---|---|---|
| (a) Hệ thống quản lý đào tạo | CSDL quan hệ SQLite `qldt.db`: `khoa`, `sinh_vien`, `hoc_phan`, `dang_ky`, `diem` | có cấu trúc, thay đổi theo giao dịch, nhiều lỗi nhập liệu |
| (b) Hệ thống quản lý học tập (LMS) | nhật ký sự kiện JSON Lines, mỗi ngày một tệp | bán cấu trúc, khối lượng lớn, có bản ghi gửi lại và dòng hỏng |
| (c) Cổng khảo sát học phần | "API" REST phân trang, có giới hạn tần suất (HTTP 429) | chỉ truy cập qua API, kéo theo trang |

Dữ liệu được làm "bẩn" có chủ đích: tên tiếng Việt lẫn dạng Unicode NFC/NFD và thừa khoảng trắng, nhiều định dạng ngày sinh và số điện thoại, hồ sơ sinh viên trùng lặp, điểm ngoài miền giá trị, mã sinh viên mồ côi, sự kiện đến trễ. Biến `SU_THAT_SINH` lưu **khóa đáp án mô phỏng** (số lỗi đã chèn) để các ô tự kiểm tra đối chiếu.

**Tăng quy mô.** Mặc định (`N_SV = 4 000`, `N_LMS = 200 000`, khoảng 50 MB JSON) cả notebook chạy vài phút. Để thử nghiệm gần với dữ liệu lớn hơn, đặt `N_SV = 40_000`, `N_LMS = 5_000_000` (khoảng 1,2 GB JSON; cần khoảng 8 GB RAM, thời gian chạy 20–40 phút) và so sánh lại các phép đo thời gian ở Phần 1, 4 và 6.

In [ ]:
# ===== 2a. Tham số, danh mục và hàm làm "bẩn" dữ liệu =====
SEED   = 42
N_SV   = 4_000       # số sinh viên (hồ sơ gốc)
N_HP   = 120         # số học phần
N_LMS  = 200_000     # số sự kiện LMS (trước khi chèn bản ghi gửi lại)
N_NGAY = 14          # số ngày nhật ký LMS (15/09/2025 – 28/09/2025)
DATA_DIR = "./data_c2"

rng = np.random.default_rng(SEED)
rnd = random.Random(SEED)
shutil.rmtree(DATA_DIR, ignore_errors=True)
for d in ["nguon/lms", "lake"]:
    os.makedirs(os.path.join(DATA_DIR, d), exist_ok=True)

KHOA = [("CNTT", "Công nghệ thông tin", 0.30), ("KT", "Kinh tế", 0.22), ("SP", "Sư phạm", 0.18),
        ("NN", "Ngoại ngữ", 0.12), ("DTVT", "Điện tử – Viễn thông", 0.10), ("LUAT", "Luật", 0.05),
        ("MT", "Môi trường", 0.03)]
MA_KHOA = [k[0] for k in KHOA]
HO = ["Nguyễn", "Trần", "Lê", "Phạm", "Hoàng", "Huỳnh", "Phan", "Vũ", "Võ", "Đặng", "Bùi", "Đỗ", "Hồ", "Ngô", "Dương", "Lý"]
DEM_NAM = ["Văn", "Minh", "Quốc", "Đức", "Hữu", "Gia", "Thanh", "Hoàng", "Anh", "Bảo"]
DEM_NU = ["Thị", "Ngọc", "Thu", "Kim", "Thanh", "Minh", "Bảo", "Mai", "Hồng", "Phương"]
TEN_NAM = ["An", "Bình", "Dũng", "Hải", "Hiếu", "Hùng", "Khánh", "Khoa", "Long", "Minh", "Nam", "Phát",
           "Phong", "Phúc", "Quân", "Sơn", "Thắng", "Trí", "Trung", "Tuấn", "Đạt", "Huy", "Lộc", "Vinh"]
TEN_NU = ["Châu", "Giang", "Hà", "Hạnh", "Hương", "Lan", "Linh", "Mai", "Ngân", "Nhung", "Quỳnh", "Tâm",
          "Thảo", "Trang", "Tú", "Uyên", "Vy", "Yến", "Hòa", "Ánh", "Diệp", "Khuê", "Ngọc", "Thư"]


def bo_dau(s):
    """Bỏ dấu tiếng Việt: 'Đặng Thị Ánh' -> 'Dang Thi Anh'."""
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.replace("đ", "d").replace("Đ", "D")


def lam_ban_ten(s, r):
    x = r.random()
    if x < 0.10: return unicodedata.normalize("NFD", s)            # gõ bằng bộ gõ tổ hợp (NFD)
    if x < 0.20: return "  " + s.replace(" ", "   ", 1) + " "      # thừa khoảng trắng
    if x < 0.26: return s.upper()
    if x < 0.32: return s.lower()
    return s


def lam_ban_ngay(d, r):
    x = r.random()
    if x < 0.70: return d.strftime("%Y-%m-%d")
    if x < 0.85: return d.strftime("%d/%m/%Y")
    if x < 0.95: return d.strftime("%d-%m-%Y")
    if x < 0.99: return d.strftime("%Y%m%d")
    return None


def lam_ban_sdt(s, r):
    x = r.random()
    if x < 0.45: return s
    if x < 0.60: return f"{s[:4]} {s[4:7]} {s[7:]}"
    if x < 0.75: return f"+84 {s[1:4]} {s[4:7]} {s[7:]}"
    if x < 0.85: return "84" + s[1:]
    if x < 0.92: return f"{s[:4]}.{s[4:7]}.{s[7:]}"
    if x < 0.95: return s[:7]                                      # thiếu chữ số -> không hợp lệ
    return None


def go_sai(s, r):
    """Gõ sai một ký tự trong tên (âm tiết cuối)."""
    parts = s.split()
    ten = parts[-1]
    k = r.randrange(len(ten))
    parts[-1] = ten[:k] + r.choice("aeiouynh") + ten[k + 1:]
    return " ".join(parts)


print("Đã khai báo tham số và hàm sinh dữ liệu.")

In [ ]:
# ===== 2b. Nguồn (a): CSDL quản lý đào tạo (SQLite) =====
# --- Hồ sơ sinh viên "đúng" (dùng làm nguồn tham chiếu và khóa đáp án mô phỏng) ---
khoa_sv = rng.choice(MA_KHOA, size=N_SV, p=[k[2] for k in KHOA])
nam_nhap = rng.choice([2022, 2023, 2024, 2025], size=N_SV)
sv_goc = []
for i in range(N_SV):
    nu = rnd.random() < 0.5
    ho_ten = f"{rnd.choice(HO)} {rnd.choice(DEM_NU if nu else DEM_NAM)} {rnd.choice(TEN_NU if nu else TEN_NAM)}"
    ns = date(int(nam_nhap[i]) - 19, 1, 1) + timedelta(days=rnd.randint(0, 729))
    sdt = rnd.choice(["03", "05", "07", "08", "09"]) + "".join(rnd.choices("0123456789", k=8))
    ma_sv = f"{nam_nhap[i] % 100:02d}{MA_KHOA.index(khoa_sv[i]) + 1:02d}{i + 1:06d}"
    sv_goc.append(dict(ma_sv=ma_sv, ho_ten=ho_ten, ngay_sinh=ns, gioi_tinh="Nữ" if nu else "Nam",
                       ma_khoa=str(khoa_sv[i]), so_dt=sdt, email=f"{ma_sv}@sv.daihoc.edu.vn",
                       nam_nhap_hoc=int(nam_nhap[i])))

# --- Bảng sinh_vien như được nhập trong hệ thống nghiệp vụ (có lỗi) ---
sv_ban, sdt_loi = [], set()
for g in sv_goc:
    sdt_b = lam_ban_sdt(g["so_dt"], rnd)
    if sdt_b is None or len(re.sub(r"\D", "", sdt_b)) < 10:
        sdt_loi.add(g["ma_sv"])
    sv_ban.append(dict(g, ho_ten=lam_ban_ten(g["ho_ten"], rnd), ngay_sinh=lam_ban_ngay(g["ngay_sinh"], rnd),
                       so_dt=sdt_b, email=g["email"].upper() if rnd.random() < 0.1 else g["email"]))

# --- Hồ sơ trùng lặp: cùng một người được nhập lại với mã mới (để đối sánh thực thể) ---
N_TRUNG = int(0.03 * N_SV)
GT_CAP_TRUNG = set()
for j, idx in enumerate(rnd.sample(range(N_SV), N_TRUNG)):
    g = sv_goc[idx]
    x = rnd.random()
    if x < 0.40:   ten = bo_dau(g["ho_ten"])                       # nhập không dấu
    elif x < 0.70: ten = lam_ban_ten(g["ho_ten"], rnd)             # khác dạng Unicode/khoảng trắng/hoa thường
    elif x < 0.85: ten = f"{g['ho_ten'].split()[0]} {g['ho_ten'].split()[-1]}"   # thiếu tên đệm
    else:          ten = go_sai(g["ho_ten"], rnd)                  # gõ sai một ký tự
    ma_moi = f"{g['nam_nhap_hoc'] % 100:02d}{MA_KHOA.index(g['ma_khoa']) + 1:02d}{N_SV + j + 1:06d}"
    ngay = None if rnd.random() < 0.10 else lam_ban_ngay(g["ngay_sinh"], rnd)
    sdt = lam_ban_sdt(g["so_dt"], rnd) if ngay is not None else g["so_dt"]
    sv_ban.append(dict(g, ma_sv=ma_moi, ho_ten=ten, ngay_sinh=ngay, so_dt=sdt, email=f"{ma_moi}@sv.daihoc.edu.vn"))
    GT_CAP_TRUNG.add(tuple(sorted((g["ma_sv"], ma_moi))))
rnd.shuffle(sv_ban)

# --- Học phần ---
TEN_HP = ["Nhập môn lập trình", "Cấu trúc dữ liệu và giải thuật", "Cơ sở dữ liệu", "Mạng máy tính", "Hệ điều hành",
          "Trí tuệ nhân tạo", "Học máy", "Xử lý dữ liệu lớn", "Kinh tế vi mô", "Kinh tế vĩ mô", "Nguyên lý kế toán",
          "Marketing căn bản", "Tâm lý học giáo dục", "Lý luận dạy học", "Ngữ âm học tiếng Anh", "Biên dịch",
          "Kỹ thuật số", "Xử lý tín hiệu số", "Luật dân sự", "Luật hành chính", "Sinh thái học", "Quản lý chất thải",
          "Xác suất thống kê", "Giải tích", "Đại số tuyến tính", "Triết học Mác – Lênin", "Tiếng Anh học thuật",
          "Kỹ năng mềm"]
khoa_hp = rng.choice(MA_KHOA, size=N_HP, p=[k[2] for k in KHOA])
hoc_phan = [dict(ma_hp=f"{khoa_hp[k]}{101 + k:03d}",
                 ten_hp=TEN_HP[k % len(TEN_HP)] + ("" if k < len(TEN_HP) else f" {k // len(TEN_HP) + 1}"),
                 so_tc=rnd.choice([2, 3, 3, 4]), ma_khoa=str(khoa_hp[k])) for k in range(N_HP)]
hp_theo_khoa = {k: [h["ma_hp"] for h in hoc_phan if h["ma_khoa"] == k] for k in MA_KHOA}

# --- Đăng ký và điểm (2 học kỳ) ---
HOC_KY = [("2024-2", datetime(2025, 6, 1)), ("2025-1", datetime(2025, 9, 1))]
LECH_KHOA = {"CNTT": -0.4, "KT": 0.2, "SP": 0.5, "NN": 0.9, "DTVT": -0.6, "LUAT": 0.3, "MT": 0.0}   # mặt bằng điểm khác nhau giữa các khoa
dk_rows, diem_rows, did = [], [], 0
for g in sv_goc:
    for hk, t0 in HOC_KY:
        chon, so_mon = set(), rnd.randint(5, 7)
        while len(chon) < so_mon:
            pool = hp_theo_khoa[g["ma_khoa"]]
            chon.add(rnd.choice(pool) if (pool and rnd.random() < 0.7) else rnd.choice(hoc_phan)["ma_hp"])
        for ma_hp in sorted(chon):
            did += 1
            dk_rows.append((did, g["ma_sv"], ma_hp, hk, (t0 - timedelta(days=rnd.randint(10, 30))).strftime("%Y-%m-%d")))
            p_thieu = 0.15 if g["ma_khoa"] == "NN" else 0.03       # thiếu điểm quá trình: MAR theo khoa
            qt = None if rnd.random() < p_thieu else round(min(10, max(0, rnd.gauss(7.0 + LECH_KHOA[g["ma_khoa"]], 1.4))), 1)
            thi = round(min(10, max(0, rnd.gauss(6.3, 1.8) if qt is None else 0.6 * qt + rnd.gauss(2.0, 1.4))), 1)
            tk = None if qt is None else round(0.4 * qt + 0.6 * thi, 1)
            upd = t0 + timedelta(days=rnd.randint(0, 27), seconds=rnd.randint(0, 86399))
            diem_rows.append([did, g["ma_sv"], ma_hp, hk, qt, thi, tk, upd.strftime("%Y-%m-%d %H:%M:%S")])

# --- Chèn lỗi có kiểm soát vào bảng điểm (ba tập rời nhau) ---
n_d = len(diem_rows)
idx_tron = rnd.sample(range(n_d), int(0.004 * n_d) + int(0.008 * n_d) * 3)
idx_khl = idx_tron[: int(0.004 * n_d)]                                       # ngoài miền [0, 10]
idx_knq = [i for i in idx_tron[int(0.004 * n_d):] if diem_rows[i][4] is not None][: int(0.008 * n_d)]  # diem_tk sai công thức
for i in idx_khl:
    r_ = diem_rows[i]
    r_[5] = rnd.choice([11.5, 12.0, -1.0, 10.5])
    r_[6] = None if r_[4] is None else round(0.4 * r_[4] + 0.6 * r_[5], 1)
for i in idx_knq:
    r_ = diem_rows[i]
    r_[6] = round(r_[6] + 1.5 if r_[6] <= 8.5 else r_[6] - 1.5, 1)
N_MO_COI = int(0.003 * n_d)                                                  # mã sinh viên không tồn tại
for k in range(N_MO_COI):
    did += 1
    qt, thi = round(rnd.uniform(4, 9), 1), round(rnd.uniform(4, 9), 1)
    diem_rows.append([did, f"99{rnd.randint(0, 99_999_999):08d}", rnd.choice(hoc_phan)["ma_hp"], "2025-1", qt, thi,
                      round(0.4 * qt + 0.6 * thi, 1), f"2025-09-{rnd.randint(1, 28):02d} 10:00:00"])

DB_PATH = f"{DATA_DIR}/nguon/qldt.db"
con = sqlite3.connect(DB_PATH)
con.executescript("""
CREATE TABLE khoa(ma_khoa TEXT PRIMARY KEY, ten_khoa TEXT);
CREATE TABLE sinh_vien(ma_sv TEXT PRIMARY KEY, ho_ten TEXT, ngay_sinh TEXT, gioi_tinh TEXT, ma_khoa TEXT,
                       so_dt TEXT, email TEXT, nam_nhap_hoc INTEGER);
CREATE TABLE hoc_phan(ma_hp TEXT PRIMARY KEY, ten_hp TEXT, so_tc INTEGER, ma_khoa TEXT);
CREATE TABLE dang_ky(id INTEGER PRIMARY KEY, ma_sv TEXT, ma_hp TEXT, hoc_ky TEXT, ngay_dk TEXT);
CREATE TABLE diem(id INTEGER PRIMARY KEY, ma_sv TEXT, ma_hp TEXT, hoc_ky TEXT, diem_qt REAL, diem_thi REAL,
                  diem_tk REAL, updated_at TEXT);
CREATE INDEX ix_diem_updated ON diem(updated_at);
""")
con.executemany("INSERT INTO khoa VALUES (?,?)", [(k[0], k[1]) for k in KHOA])
con.executemany("INSERT INTO sinh_vien VALUES (?,?,?,?,?,?,?,?)",
                [(s["ma_sv"], s["ho_ten"], s["ngay_sinh"], s["gioi_tinh"], s["ma_khoa"], s["so_dt"], s["email"],
                  s["nam_nhap_hoc"]) for s in sv_ban])
con.executemany("INSERT INTO hoc_phan VALUES (?,?,?,?)", [tuple(h.values()) for h in hoc_phan])
con.executemany("INSERT INTO dang_ky VALUES (?,?,?,?,?)", dk_rows)
con.executemany("INSERT INTO diem VALUES (?,?,?,?,?,?,?,?)", diem_rows)
con.commit(); con.close()

# Hồ sơ gốc đã xác minh cho 10% sinh viên: nguồn tham chiếu để đo chiều "chính xác"
tham_chieu = rnd.sample(sv_goc, N_SV // 10)
pd.DataFrame([{"ma_sv": g["ma_sv"], "ho_ten": g["ho_ten"], "ngay_sinh": g["ngay_sinh"].isoformat(),
               "so_dt": g["so_dt"]} for g in tham_chieu]).to_csv(f"{DATA_DIR}/nguon/tham_chieu_ho_so.csv", index=False)

print(f"qldt.db: {len(sv_ban)} sinh viên ({N_TRUNG} hồ sơ trùng), {N_HP} học phần, "
      f"{len(dk_rows)} đăng ký, {len(diem_rows)} dòng điểm")

In [ ]:
# ===== 2c. Nguồn (b): nhật ký sự kiện LMS dạng JSON Lines =====
LOAI = ["dang_nhap", "xem_video", "lam_quiz", "nop_bai", "dien_dan"]
P_LOAI = [0.25, 0.35, 0.18, 0.10, 0.12]
w_gio = np.array([1, .5, .3, .2, .2, .5, 1.5, 3, 5, 6, 6, 5, 3, 3, 5, 6, 5, 4, 3, 4, 6, 6, 4, 2], dtype=float)

ma_sv_goc = np.array([g["ma_sv"] for g in sv_goc])
hp_hk1 = {}
for (_, ma_sv, ma_hp, hk, _) in dk_rows:
    if hk == "2025-1":
        hp_hk1.setdefault(ma_sv, []).append(ma_hp)

w_sv = rng.gamma(2.0, 1.0, N_SV); w_sv /= w_sv.sum()          # mức độ hoạt động khác nhau giữa sinh viên
idx_sv = rng.choice(N_SV, size=N_LMS, p=w_sv)
loai = rng.choice(LOAI, size=N_LMS, p=P_LOAI)
ngay_i = rng.integers(0, N_NGAY, N_LMS)
giay_tu_0h = ngay_i * 86400 + rng.choice(24, size=N_LMS, p=w_gio / w_gio.sum()) * 3600 + rng.integers(0, 3600, N_LMS)
t_su_kien = np.datetime64("2025-09-15T00:00:00") + giay_tu_0h.astype("timedelta64[s]")
tre = np.round(rng.lognormal(3.0, 1.0, N_LMS)).astype(int)     # độ trễ nạp (giây)
tre_muon = rng.random(N_LMS) < 0.03                            # thiết bị ngoại tuyến gửi bù sau 1–6 giờ
tre[tre_muon] += rng.integers(3600, 6 * 3600, tre_muon.sum())
t_nap = t_su_kien + tre.astype("timedelta64[s]")
thoi_luong = np.round(rng.lognormal(5.3, 0.9, N_LMS)).astype(int)
tl_null = rng.random(N_LMS) < 0.03
tl_ngoai_lai = (~tl_null) & (rng.random(N_LMS) < 0.004)        # phiên bỏ quên không đăng xuất
thoi_luong[tl_ngoai_lai] = rng.integers(20_000, 86_400, tl_ngoai_lai.sum())
quiz = np.round(np.clip(rng.normal(7, 2, N_LMS), 0, 10), 1)
thiet_bi = rng.choice(["web", "android", "ios"], size=N_LMS, p=[0.5, 0.3, 0.2])
tk_khach = rng.random(N_LMS) < 0.003                           # tài khoản giảng viên/khách: không có trong danh mục SV
chon_hp = rng.integers(0, 1000, N_LMS)
s_su_kien = np.datetime_as_string(t_su_kien, unit="s")
s_nap = np.datetime_as_string(t_nap, unit="s")

theo_ngay = {k: [] for k in range(N_NGAY)}
ban_ghi_lms = []
for i in range(N_LMS):
    sv = ma_sv_goc[idx_sv[i]]
    ds_hp = hp_hk1[sv]
    rec = {"event_id": f"ev{i + 1:08d}", "ma_sv": f"GV{idx_sv[i]:05d}" if tk_khach[i] else sv,
           "ma_hp": ds_hp[chon_hp[i] % len(ds_hp)], "loai": loai[i], "thoi_gian": s_su_kien[i],
           "thoi_gian_nap": s_nap[i], "thoi_luong_giay": None if tl_null[i] else int(thoi_luong[i]),
           "thiet_bi": thiet_bi[i], "diem_quiz": float(quiz[i]) if loai[i] == "lam_quiz" else None}
    ban_ghi_lms.append(rec)
    theo_ngay[int(ngay_i[i])].append(json.dumps(rec, ensure_ascii=False))

# Gửi lại theo ngữ nghĩa "ít nhất một lần": 2% sự kiện xuất hiện hai lần (cùng event_id, thời điểm nạp muộn hơn)
idx_dup = rng.choice(N_LMS, size=int(0.02 * N_LMS), replace=False)
them_giay = rng.integers(5, 120, len(idx_dup))
for i, k in zip(idx_dup, them_giay):
    rec = dict(ban_ghi_lms[i], thoi_gian_nap=str(np.datetime_as_string(t_nap[i] + np.timedelta64(int(k), "s"), unit="s")))
    theo_ngay[int(ngay_i[i])].append(json.dumps(rec, ensure_ascii=False))
# Dòng hỏng (mất kết nối giữa chừng): JSON bị cắt cụt
N_HONG = 40
for k in range(N_HONG):
    s = json.dumps(dict(ban_ghi_lms[rnd.randrange(N_LMS)], event_id=f"evx{k:05d}"), ensure_ascii=False)
    theo_ngay[rnd.randrange(N_NGAY)].append(s[: rnd.randint(20, len(s) - 10)])

for k, lines in theo_ngay.items():
    rnd.shuffle(lines)
    ngay_str = (date(2025, 9, 15) + timedelta(days=k)).isoformat()
    with open(f"{DATA_DIR}/nguon/lms/lms_{ngay_str}.jsonl", "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

tre_tat_ca = np.concatenate([tre, tre[idx_dup] + them_giay])
tl_null_tat_ca = np.concatenate([tl_null, tl_null[idx_dup]])
kich_thuoc = sum(os.path.getsize(os.path.join(f"{DATA_DIR}/nguon/lms", f)) for f in os.listdir(f"{DATA_DIR}/nguon/lms"))
print(f"LMS: {N_LMS + len(idx_dup) + N_HONG} dòng trong {N_NGAY} tệp, {kich_thuoc / 1e6:.1f} MB")
print("Ví dụ một dòng:", theo_ngay[0][0][:200])

In [ ]:
# ===== 2d. Nguồn (c): API cổng khảo sát học phần (phân trang, giới hạn tần suất) =====
class LoiGioiHanTanSuat(Exception):
    """Mô phỏng phản hồi HTTP 429 Too Many Requests."""


class CongKhaoSatAPI:
    """API REST mô phỏng: GET /khao-sat?page=..&page_size=.. trả về JSON có trường next_page."""
    TRANG_TOI_DA = 500

    def __init__(self, ban_ghi, ti_le_429=0.15, seed=SEED):
        self._data = sorted(ban_ghi, key=lambda r: r["id"])
        self._r = random.Random(seed)
        self.ti_le_429 = ti_le_429
        self.so_lan_goi = 0
        self.so_lan_429 = 0

    def get(self, page=1, page_size=100):
        self.so_lan_goi += 1
        if page_size > self.TRANG_TOI_DA:
            raise ValueError(f"page_size tối đa {self.TRANG_TOI_DA}")
        if self._r.random() < self.ti_le_429:
            self.so_lan_429 += 1
            raise LoiGioiHanTanSuat("HTTP 429: vượt giới hạn tần suất, hãy thử lại sau")
        bat_dau = (page - 1) * page_size
        trang = self._data[bat_dau: bat_dau + page_size]
        return {"page": page, "page_size": page_size, "total": len(self._data),
                "next_page": page + 1 if bat_dau + page_size < len(self._data) else None,
                "data": [dict(r) for r in trang]}


NHAN_XET = ["", "", "Bài giảng dễ hiểu", "Cần thêm bài tập thực hành", "Tốc độ giảng hơi nhanh",
            "Tài liệu đầy đủ", "Nên có thêm ví dụ thực tế"]
khao_sat, kid = [], 0
for r_ in diem_rows:
    if r_[3] != "2025-1" or r_[1].startswith("99"):
        continue
    diem_ref = r_[6] if r_[6] is not None else r_[5]
    if rnd.random() < min(0.9, 0.15 + 0.05 * max(0.0, diem_ref)):  # người điểm thấp ít phản hồi (MNAR)
        kid += 1
        khao_sat.append({"id": kid, "ma_sv": r_[1], "ma_hp": r_[2], "hoc_ky": "2025-1",
                         "muc_hai_long": int(min(5, max(1, round(rnd.gauss(2 + 0.3 * min(diem_ref, 10), 0.8))))),
                         "nhan_xet": rnd.choice(NHAN_XET),
                         "thoi_gian_gui": f"2025-09-{rnd.randint(20, 28):02d}T{rnd.randint(7, 22):02d}:{rnd.randint(0, 59):02d}:00"})
api_khao_sat = CongKhaoSatAPI(khao_sat)

# ===== Khóa đáp án mô phỏng (dùng trong các ô tự kiểm tra) =====
SU_THAT_SINH = {
    "n_sv_ban_ghi": len(sv_ban), "n_cap_trung": len(GT_CAP_TRUNG), "n_diem": len(diem_rows),
    "n_diem_ngoai_mien": len(idx_khl), "n_diem_khong_nhat_quan": len(idx_knq), "n_diem_mo_coi": N_MO_COI,
    "n_lms_hop_le": N_LMS + len(idx_dup), "n_lms_trung": len(idx_dup), "n_lms_hong": N_HONG,
    "n_lms_khach": int(tk_khach.sum() + tk_khach[idx_dup].sum()),
    "ty_le_kip_thoi_15p": float(np.mean(tre_tat_ca <= 900)),
    "ty_le_day_du_thoi_luong": float(1 - tl_null_tat_ca.mean()),
    "ty_le_chinh_xac_sdt": 1 - len({g["ma_sv"] for g in tham_chieu} & sdt_loi) / len(tham_chieu),
    "n_khao_sat": len(khao_sat),
}
print("API khảo sát:", len(khao_sat), "phản hồi (chỉ truy cập được qua api_khao_sat.get)")
pd.Series(SU_THAT_SINH, name="giá trị").to_frame()

---
## Phần 1. Thu thập dữ liệu đa nguồn

### Khái niệm từ bài giảng
*Slide mục 2.1: "Thu thập dữ liệu: khái niệm", "Bốn phương thức thu thập dữ liệu", "So sánh các phương thức thu thập", "Đánh đổi giữa các phương thức thu thập"; mục 2.2: "Apache Sqoop", "Định dạng dữ liệu và tuần tự hóa".*

- **Thu thập dữ liệu** (data acquisition/ingestion) là quá trình xác định, truy cập và đưa dữ liệu từ nguồn vào hệ thống lưu trữ/xử lý, kèm **siêu dữ liệu** (metadata) về nguồn gốc, thời điểm, lược đồ.
- Bốn phương thức: **kéo theo lô** (batch pull: hệ đích truy vấn nguồn theo chu kỳ $T$), **đẩy** (push: nguồn chủ động gửi), **luồng** (streaming: sự kiện qua lớp đệm bền vững như Kafka), **thu thập dữ liệu thay đổi** (change data capture, CDC: đọc nhật ký giao dịch).

| Tiêu chí | Kéo theo lô | Đẩy | Luồng | CDC |
|---|---|---|---|---|
| Độ trễ | phút – giờ (theo $T$) | giây | mili giây – giây | giây |
| Tải lên nguồn | cao khi quét bảng lớn | do nguồn kiểm soát | thấp | thấp |
| Phát hiện bản ghi bị xóa | khó | tùy ứng dụng | tùy thiết kế sự kiện | có |
| Công cụ tiêu biểu | Sqoop, Spark JDBC | HTTP, Flume | Kafka, NiFi | Debezium, Kafka Connect |

- Với kéo theo lô chu kỳ $T$ và thời gian xử lý $t_p$: độ trễ trung bình $\bar L \approx T/2 + t_p$, tối đa $L_{\max} \approx T + t_p$.
- Sqoop/Spark JDBC đọc song song bằng cách **chia khoảng** cột khóa: $\Delta = (\max - \min)/m$; khóa phân bố lệch làm các tác vụ mất cân bằng.
- **Lược đồ khai báo** (schema-on-read có kiểm soát) tránh một lượt quét suy luận kiểu và cho phép tách dòng hỏng vào cột `_corrupt_record`.

Trong notebook, SQLite đóng vai MySQL/Oracle; pandas → Spark đóng vai bộ nạp JDBC.

### Ví dụ mẫu 1a — Kéo theo lô từ CSDL quan hệ
Hàm `doc_bang_sqlite` thực hiện một truy vấn trên nguồn, chuyển kết quả sang Spark DataFrame với **lược đồ khai báo**. Trên cụm thật, bước này dùng Spark JDBC đọc song song theo `partitionColumn` hoặc Sqoop:

```python
# Spark JDBC: 16 truy vấn song song theo khoảng id (lowerBound/upperBound chỉ quyết định bước chia, không lọc dữ liệu)
df = (spark.read.format("jdbc")
      .option("url", "jdbc:mysql://db-daotao:3306/qldt").option("dbtable", "diem")
      .option("user", "etl").option("password", secret)          # lấy từ kho bí mật
      .option("partitionColumn", "id").option("lowerBound", 1).option("upperBound", 5_000_000)
      .option("numPartitions", 16).option("fetchsize", 10_000).load())
```
```bash
sqoop import --connect jdbc:mysql://db-daotao:3306/qldt --table diem --split-by id -m 8 \
  --as-parquetfile --target-dir /lake/raw/qldt/diem --password-file /user/etl/.pw
```

In [ ]:
def doc_bang_sqlite(sql, schema, params=()):
    """Kéo theo lô: truy vấn CSDL nguồn -> pandas -> Spark DataFrame với lược đồ khai báo."""
    con = sqlite3.connect(DB_PATH)
    try:
        pdf = pd.read_sql_query(sql, con, params=params)
    finally:
        con.close()
    pdf = pdf.astype(object).where(pdf.notna(), None)      # NaN của pandas -> NULL của Spark
    return spark.createDataFrame(pdf, schema=schema)


SCHEMA_SV = T.StructType([T.StructField(c, T.StringType()) for c in
                          ["ma_sv", "ho_ten", "ngay_sinh", "gioi_tinh", "ma_khoa", "so_dt", "email"]]
                         + [T.StructField("nam_nhap_hoc", T.IntegerType())])
SCHEMA_DIEM = T.StructType([T.StructField("id", T.LongType()), T.StructField("ma_sv", T.StringType()),
                            T.StructField("ma_hp", T.StringType()), T.StructField("hoc_ky", T.StringType()),
                            T.StructField("diem_qt", T.DoubleType()), T.StructField("diem_thi", T.DoubleType()),
                            T.StructField("diem_tk", T.DoubleType()), T.StructField("updated_at", T.StringType())])
SCHEMA_HP = "ma_hp string, ten_hp string, so_tc int, ma_khoa string"

t0 = time.perf_counter()
sv_raw = doc_bang_sqlite("SELECT * FROM sinh_vien", SCHEMA_SV).cache()
diem_raw = doc_bang_sqlite("SELECT * FROM diem", SCHEMA_DIEM).withColumn("updated_at", F.to_timestamp("updated_at")).cache()
hp_raw = doc_bang_sqlite("SELECT * FROM hoc_phan", SCHEMA_HP).cache()
print(f"sinh_vien: {sv_raw.count()} dòng | diem: {diem_raw.count()} dòng | hoc_phan: {hp_raw.count()} dòng "
      f"| {time.perf_counter() - t0:.2f} s")
sv_raw.show(6, truncate=False)
diem_raw.printSchema()

### Ví dụ mẫu 1b — Nhật ký LMS: lược đồ khai báo so với suy luận lược đồ
Đọc JSON Lines với lược đồ khai báo, chế độ `PERMISSIVE` và cột `_corrupt_record` để giữ lại dòng hỏng (sẽ đưa vào vùng cách ly ở Phần 6). So sánh với việc để Spark tự suy luận lược đồ.

In [ ]:
DUONG_DAN_LMS = f"{DATA_DIR}/nguon/lms/*.jsonl"
SCHEMA_LMS = T.StructType([
    T.StructField("event_id", T.StringType()), T.StructField("ma_sv", T.StringType()),
    T.StructField("ma_hp", T.StringType()), T.StructField("loai", T.StringType()),
    T.StructField("thoi_gian", T.TimestampType()), T.StructField("thoi_gian_nap", T.TimestampType()),
    T.StructField("thoi_luong_giay", T.IntegerType()), T.StructField("thiet_bi", T.StringType()),
    T.StructField("diem_quiz", T.DoubleType()), T.StructField("_corrupt_record", T.StringType())])

# (1) Suy luận lược đồ: Spark phải quét toàn bộ dữ liệu một lượt chỉ để đoán kiểu
t0 = time.perf_counter()
df_suy_luan = spark.read.json(DUONG_DAN_LMS)
n_suy_luan = df_suy_luan.count()
t_suy_luan = time.perf_counter() - t0

# (2) Lược đồ khai báo: không cần lượt quét suy luận, kiểu thời gian được phân tích ngay khi đọc
t0 = time.perf_counter()
lms_raw = (spark.read.schema(SCHEMA_LMS)
           .option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record")
           .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
           .json(DUONG_DAN_LMS)
           .cache())
n_khai_bao = lms_raw.count()
t_khai_bao = time.perf_counter() - t0

lms_hop_le = lms_raw.where(F.col("_corrupt_record").isNull()).drop("_corrupt_record").cache()
n_hong = n_khai_bao - lms_hop_le.count()
print(pd.DataFrame({"cách đọc": ["suy luận lược đồ", "lược đồ khai báo"], "số dòng": [n_suy_luan, n_khai_bao],
                    "thời gian (s)": [round(t_suy_luan, 2), round(t_khai_bao, 2)]}).to_string(index=False))
print("Kiểu cột thoi_gian khi suy luận:", dict(df_suy_luan.dtypes)["thoi_gian"], "| khi khai báo: timestamp")
print("Số dòng JSON hỏng tách ra:", n_hong)
lms_hop_le.show(3, truncate=False)

### Ví dụ mẫu 1c — Gọi API phân trang
Mỗi lần gọi trả về một trang và trường `next_page`; API có thể trả lỗi 429 khi gọi quá nhanh.

In [ ]:
try:
    trang = api_khao_sat.get(page=1, page_size=2)
    print(json.dumps(trang, ensure_ascii=False, indent=1))
except LoiGioiHanTanSuat as e:
    print("Lần gọi này gặp lỗi:", e, "→ cần cơ chế thử lại (Bài tập 1.2)")

### Bài tập 1.1 — Chia khoảng khóa để đọc song song (mô phỏng `--split-by`)
1. Viết `chia_khoang(lo, hi, m)` trả về $m$ cặp $(a, b)$ liên tiếp phủ đoạn $[lo, hi]$ với bước $\Delta = (hi - lo)/m$ (biên thứ $k$ là `lo + int(k*Δ)`, biên cuối là `hi`).
2. Hoàn thiện vòng lặp trong `doc_song_song`: mỗi khoảng là một "tác vụ map" đọc `WHERE cot >= a AND cot < b` (khoảng cuối dùng `<=`), ghi lại số dòng của từng khoảng, rồi hợp nhất các phần.

Ô kiểm tra sẽ đo **độ lệch tải** $\max/\text{trung bình}$ khi chia theo `id` và khi chia theo thời điểm cập nhật (epoch giây).

In [ ]:
# BÀI TẬP 1.1 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def chia_khoang(lo, hi, m):
    """Trả về m cặp (a, b): khoảng k là a <= cot < b; khoảng cuối a <= cot <= b."""
    # Gợi ý: buoc = (hi - lo) / m ; bien = [lo + int(k * buoc) for k in range(m)] + [hi]
    ...
    return None


def doc_song_song(bang, cot, m, schema):
    """Đọc `bang` bằng m tác vụ, mỗi tác vụ một khoảng của biểu thức `cot`. Trả về (DataFrame, số dòng mỗi khoảng)."""
    con = sqlite3.connect(DB_PATH)
    lo, hi = con.execute(f"SELECT MIN({cot}), MAX({cot}) FROM {bang}").fetchone()
    con.close()
    phan, so_dong = [], []
    for k, (a, b) in enumerate(chia_khoang(lo, hi, m)):
        toan_tu = "<=" if k == m - 1 else "<"
        # Gợi ý: df_k = doc_bang_sqlite(f"SELECT * FROM {bang} WHERE {cot} >= ? AND {cot} {toan_tu} ?", schema, params=(a, b))
        df_k = ...
        phan.append(df_k)
        so_dong.append(...)
    return None, so_dong   # Gợi ý: reduce(DataFrame.unionByName, phan)

In [ ]:
# Ô TỰ KIỂM TRA
kh = chia_khoang(1, 1000, 4)
assert kh is not None and len(kh) == 4, "chia_khoang phải trả về đúng m khoảng"
assert kh[0][0] == 1 and kh[-1][1] == 1000 and all(kh[i][1] == kh[i + 1][0] for i in range(3)), "các khoảng phải liên tiếp, phủ [lo, hi]"

df_ss, so_dong_id = doc_song_song("diem", "id", 8, SCHEMA_DIEM)
n_diem = diem_raw.count()
assert df_ss.count() == n_diem and df_ss.select("id").distinct().count() == n_diem, "đọc song song phải đủ và không trùng"
EPOCH = "CAST(strftime('%s', updated_at) AS INTEGER)"
_, so_dong_tg = doc_song_song("diem", EPOCH, 8, SCHEMA_DIEM)
assert sum(so_dong_tg) == n_diem
lech_id, lech_tg = max(so_dong_id) / np.mean(so_dong_id), max(so_dong_tg) / np.mean(so_dong_tg)
print("Số dòng mỗi khoảng (theo id)        :", so_dong_id, f"→ lệch {lech_id:.2f}")
print("Số dòng mỗi khoảng (theo updated_at):", so_dong_tg, f"→ lệch {lech_tg:.2f}")
assert lech_id < 1.2 and lech_tg > 1.5
dat("Bài tập 1.1")

### Bài tập 1.2 — Kéo toàn bộ dữ liệu từ API phân trang, có thử lại khi gặp HTTP 429
Hoàn thiện `thu_thap_api`: lặp qua các trang theo `next_page`; khi gặp `LoiGioiHanTanSuat`, chờ theo **lùi lũy thừa** $t_k = t_0 \cdot 2^k + \text{jitter}$ rồi thử lại (tối đa `so_lan_thu` lần cho mỗi trang). Sau đó ghi các bản ghi vào **vùng thô** dạng JSON Lines (giữ nguyên định dạng nguồn) và đọc lại bằng Spark.

In [ ]:
# BÀI TẬP 1.2 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def thu_thap_api(api, page_size=500, so_lan_thu=6, t0=0.01):
    """Kéo toàn bộ dữ liệu từ API phân trang; thử lại với lùi lũy thừa khi gặp HTTP 429."""
    ket_qua, page = [], 1
    while page is not None:
        # Gợi ý: vòng for k in range(so_lan_thu): try: resp = api.get(...); break
        #        except LoiGioiHanTanSuat: time.sleep(t0 * 2 ** k + random.uniform(0, t0))
        #        (dùng for ... else: raise RuntimeError(...) khi hết số lần thử)
        ...
        page = None   # Gợi ý: nối resp["data"] vào ket_qua, chuyển sang resp["next_page"]
    return ket_qua


ban_ghi_ks = thu_thap_api(api_khao_sat)
THU_MUC_KS = f"{DATA_DIR}/lake/raw/khao_sat/ngay_nap=2025-09-29"
os.makedirs(THU_MUC_KS, exist_ok=True)
# Gợi ý: ghi mỗi bản ghi một dòng json.dumps(r, ensure_ascii=False) vào f"{THU_MUC_KS}/part-00000.jsonl"
...
ks_raw = None   # Gợi ý: spark.read.schema("id long, ma_sv string, ...").json(THU_MUC_KS)

In [ ]:
# Ô TỰ KIỂM TRA
assert isinstance(ban_ghi_ks, list) and len(ban_ghi_ks) == SU_THAT_SINH["n_khao_sat"], "phải thu đủ mọi bản ghi"
ids = [r["id"] for r in ban_ghi_ks]
assert len(set(ids)) == len(ids), "không được trùng bản ghi giữa các trang"
assert api_khao_sat.so_lan_429 > 0, "kịch bản mô phỏng luôn có lỗi 429; hàm phải vượt qua được"
assert ks_raw is not None and ks_raw.count() == len(ban_ghi_ks)
dat("Bài tập 1.2")

### Bài tập 1.3 — Độ trễ dữ liệu khi kéo theo lô
Lô chạy tại các thời điểm $T, 2T, 3T, \dots$ (phút); một sự kiện phát sinh lúc $t$ được nạp ở lô kế tiếp và sẵn sàng sau thêm $t_p$ phút. Viết `do_tre_keo_lo(t_phut, T, t_p)` trả về mảng độ trễ (phút) của từng sự kiện. Ô kiểm tra so sánh với công thức trên slide và vẽ đánh đổi giữa độ trễ và số lần truy vấn nguồn mỗi ngày.

In [ ]:
# Thời điểm phát sinh của các sự kiện LMS, tính bằng phút kể từ 00:00 ngày 15/09/2025
t_phut = (lms_hop_le
          .select(((F.unix_timestamp("thoi_gian") - F.unix_timestamp(F.lit("2025-09-15 00:00:00"))) / 60).alias("t"))
          .toPandas()["t"].to_numpy())
print(len(t_phut), "sự kiện; phút nhỏ nhất/lớn nhất:", t_phut.min(), t_phut.max())

In [ ]:
# BÀI TẬP 1.3 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def do_tre_keo_lo(t_phut, T, t_p):
    """Độ trễ (phút) = thời điểm lô kế tiếp - thời điểm phát sinh + thời gian xử lý."""
    # Gợi ý: thời điểm lô kế tiếp = np.ceil(t_phut / T) * T
    return None

In [ ]:
# Ô TỰ KIỂM TRA
t_p = 5
ds_T = [15, 30, 60, 120, 240, 720, 1440]
bang = []
for T_ in ds_T:
    L = do_tre_keo_lo(t_phut, T_, t_p)
    assert L is not None and len(L) == len(t_phut)
    assert L.max() <= T_ + t_p + 1e-9 and L.min() >= t_p - 1e-9
    bang.append({"T (phút)": T_, "truy vấn/ngày": 1440 // T_, "L trung bình": round(L.mean(), 1),
                 "T/2 + t_p": T_ / 2 + t_p, "L tối đa": round(L.max(), 1)})
bang = pd.DataFrame(bang)
print(bang.to_string(index=False))
nho = bang["T (phút)"] <= 240
assert np.allclose(bang.loc[nho, "L trung bình"], bang.loc[nho, "T/2 + t_p"], rtol=0.03), "với T nhỏ, độ trễ trung bình phải xấp xỉ T/2 + t_p"
print("Lưu ý: với T = 720 và 1440 phút, công thức T/2 + t_p không còn đúng vì sự kiện tập trung vào ban ngày (không phân bố đều trong chu kỳ).")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(bang["truy vấn/ngày"], bang["L trung bình"], "o-", color="#8c2d04")
for _, r in bang.iterrows():
    ax.annotate(f"T={int(r['T (phút)'])}", (r["truy vấn/ngày"], r["L trung bình"]), textcoords="offset points", xytext=(5, 5), fontsize=10)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Số lần truy vấn nguồn mỗi ngày (tải lên nguồn)"); ax.set_ylabel("Độ trễ trung bình (phút)")
ax.set_title("Đánh đổi độ trễ – tải nguồn khi kéo theo lô (dữ liệu mô phỏng)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
dat("Bài tập 1.3")

### Câu hỏi thảo luận — Phần 1
1. Ở Bài tập 1.1, vì sao chia theo `updated_at` gây lệch tải lớn? Đề xuất cách chọn biên khoảng theo phân vị (ví dụ dùng `approxQuantile` rồi truyền `predicates` cho Spark JDBC) và phân tích chi phí của bước tính phân vị đó trên CSDL nguồn.
2. Lược đồ khai báo nhanh hơn và an toàn hơn suy luận lược đồ, nhưng điều gì xảy ra khi nguồn LMS thêm trường mới hoặc đổi kiểu `thoi_luong_giay` sang chuỗi? Thiết kế cơ chế phát hiện **trôi lược đồ** (schema drift) ở tầng thu nhận.
3. Ở Bài tập 1.3, vì sao độ trễ trung bình với $T = 720$ và $T = 1440$ phút lệch khỏi $T/2 + t_p$? Nếu lô đêm chạy lúc 0 giờ, nên dời giờ chạy thế nào để giảm độ trễ trung bình mà không tăng tải giờ cao điểm?
4. Từ biểu đồ Bài tập 1.3: nếu yêu cầu độ tươi của dashboard cảnh báo học vụ là 10 phút, kéo theo lô còn phù hợp không? So sánh với CDC về tải nguồn và độ phức tạp vận hành.

---
## Phần 2. Thu thập dữ liệu thay đổi (CDC) và nạp gia tăng

### Khái niệm từ bài giảng
*Slide mục 2.2: "Sqoop: import và nạp gia tăng", "Thu thập dữ liệu thay đổi (CDC)", "CDC: so sánh các cách tiếp cận"; Câu hỏi thảo luận 1 của chương.*

| Cách tiếp cận | Bắt thao tác xóa | Tải nguồn | Độ trễ |
|---|---|---|---|
| Truy vấn theo cột thời gian (`--incremental lastmodified`) | không | cao | phút |
| So sánh hai ảnh chụp (snapshot diff, khóa + băm bản ghi) | có | rất cao (quét toàn bộ) | theo chu kỳ ảnh chụp |
| Đọc nhật ký giao dịch (Debezium) | có | thấp | giây |

- `sqoop import --incremental lastmodified --check-column updated_at --last-value ... --merge-key id` chỉ đọc các dòng có `updated_at` lớn hơn **dấu mốc** (watermark) rồi hợp nhất theo khóa. Phương pháp này **không thấy** dòng bị xóa và dòng bị sửa mà không cập nhật `updated_at`.
- Sự kiện CDC mang **loại thao tác** (`c`/`u`/`d`) cùng trạng thái trước/sau, ví dụ `{"op":"u","before":{"id":7,"diem":6.5},"after":{"id":7,"diem":7.0}}`.
- Áp dụng thay đổi vào đích phải **lũy đẳng** (idempotent): áp dụng lại cùng một tập thay đổi không làm sai kết quả.

### Ví dụ mẫu 2a — Ảnh chụp ngày 30/09 và mô phỏng thay đổi ở hệ nguồn
Bảng `diem` vừa đọc ở Phần 1 là **ảnh chụp 1**. Hàm dưới đây mô phỏng một ngày giao dịch trên hệ nguồn: phúc khảo (sửa có cập nhật `updated_at`), script sửa lỗi hàng loạt **không** cập nhật `updated_at` ("sửa ngầm"), hủy kết quả và bổ sung điểm.

In [ ]:
RAW = f"{DATA_DIR}/lake/raw"
snap1 = diem_raw
snap1.write.mode("overwrite").parquet(f"{RAW}/qldt/diem/ngay_nap=2025-09-30")
# Dấu mốc lấy dưới dạng chuỗi ngay trong Spark để tránh lệch múi giờ khi chuyển sang Python
MOC_NUOC = snap1.agg(F.date_format(F.max("updated_at"), "yyyy-MM-dd HH:mm:ss")).first()[0]
print("Dấu mốc (watermark) sau lần nạp đầy đủ:", MOC_NUOC)


def mo_phong_thay_doi_nguon(seed=SEED + 2):
    r = random.Random(seed)
    con = sqlite3.connect(DB_PATH)
    ung_vien = [x[0] for x in con.execute(
        "SELECT id FROM diem WHERE hoc_ky='2025-1' AND diem_qt IS NOT NULL AND diem_qt <= 9.5 "
        "AND diem_thi BETWEEN 0 AND 9 ORDER BY id")]
    r.shuffle(ung_vien)
    id_sua, id_sua_ngam, id_xoa = ung_vien[:600], ung_vien[600:680], ung_vien[680:830]
    t = datetime(2025, 10, 1, 8, 0, 0)
    for k, i in enumerate(id_sua):            # phúc khảo: +0,5 điểm thi, cập nhật updated_at
        con.execute("UPDATE diem SET diem_thi = ROUND(diem_thi + 0.5, 1), "
                    "diem_tk = ROUND(0.4*diem_qt + 0.6*(diem_thi + 0.5), 1), updated_at = ? WHERE id = ?",
                    ((t + timedelta(minutes=k)).strftime("%Y-%m-%d %H:%M:%S"), i))
    for i in id_sua_ngam:                     # sửa hàng loạt nhưng KHÔNG cập nhật updated_at
        con.execute("UPDATE diem SET diem_qt = ROUND(diem_qt + 0.5, 1), "
                    "diem_tk = ROUND(0.4*(diem_qt + 0.5) + 0.6*diem_thi, 1) WHERE id = ?", (i,))
    con.executemany("DELETE FROM diem WHERE id = ?", [(i,) for i in id_xoa])
    max_id = con.execute("SELECT MAX(id) FROM diem").fetchone()[0]
    id_moi = list(range(max_id + 1, max_id + 401))
    for k, i in enumerate(id_moi):            # bổ sung điểm học phần nhập muộn
        g = sv_goc[r.randrange(N_SV)]
        qt, thi = round(r.uniform(4, 10), 1), round(r.uniform(3, 10), 1)
        con.execute("INSERT INTO diem VALUES (?,?,?,?,?,?,?,?)",
                    (i, g["ma_sv"], r.choice(hoc_phan)["ma_hp"], "2025-1", qt, thi, round(0.4 * qt + 0.6 * thi, 1),
                     (t + timedelta(hours=2, seconds=k)).strftime("%Y-%m-%d %H:%M:%S")))
    con.commit(); con.close()
    return {"c": set(id_moi), "u": set(id_sua) | set(id_sua_ngam), "d": set(id_xoa), "u_ngam": set(id_sua_ngam)}


SU_THAT_CDC = mo_phong_thay_doi_nguon()
snap2 = doc_bang_sqlite("SELECT * FROM diem", SCHEMA_DIEM).withColumn("updated_at", F.to_timestamp("updated_at")).cache()
print({k: len(v) for k, v in SU_THAT_CDC.items()}, "| ảnh chụp 2:", snap2.count(), "dòng")

### Ví dụ mẫu 2b — Nạp gia tăng theo cột `updated_at` (tương đương `sqoop --incremental lastmodified --merge-key id`)

In [ ]:
t0 = time.perf_counter()
delta = (doc_bang_sqlite("SELECT * FROM diem WHERE updated_at > ?", SCHEMA_DIEM, params=(MOC_NUOC,))
         .withColumn("updated_at", F.to_timestamp("updated_at")))
w_moi_nhat = Window.partitionBy("id").orderBy(F.col("updated_at").desc())
diem_gia_tang = (snap1.unionByName(delta)
                 .withColumn("rn", F.row_number().over(w_moi_nhat)).where("rn = 1").drop("rn").cache())
n_gia_tang = diem_gia_tang.count()
t_gia_tang = time.perf_counter() - t0

con_song_o_dich = diem_gia_tang.join(snap2, "id", "left_anti").count()          # đã xóa ở nguồn nhưng còn ở đích
lech_gia_tri = diem_gia_tang.exceptAll(snap2).join(snap2.select("id"), "id").count()  # sửa nhưng đích không biết
print(f"Đọc từ nguồn {delta.count()} dòng (thay vì {snap2.count()}), thời gian {t_gia_tang:.2f} s")
print(pd.DataFrame({"chỉ số": ["dòng bị xóa ở nguồn nhưng còn ở đích", "dòng sửa ngầm bị bỏ sót"],
                    "giá trị": [con_song_o_dich, lech_gia_tri],
                    "sự thật mô phỏng": [len(SU_THAT_CDC["d"]), len(SU_THAT_CDC["u_ngam"])]}).to_string(index=False))

### Bài tập 2.1 — Phát hiện thay đổi bằng so sánh hai ảnh chụp (khóa + băm bản ghi)
Viết `phat_hien_thay_doi(cu, moi, khoa, cot)` trả về DataFrame gồm cột khóa, cột `op` ∈ {`c`, `u`, `d`} và các cột của bản ghi mới (null khi `op = 'd'`). Dùng hàm băm SHA-256 trên các cột giá trị; lưu ý phân biệt `NULL` với chuỗi rỗng và giữ đúng vị trí cột.

In [ ]:
# BÀI TẬP 2.1 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def phat_hien_thay_doi(cu, moi, khoa="id", cot=None):
    cot = cot or [c for c in moi.columns if c != khoa]
    # Gợi ý 1: bam = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("∅")) for c in cot]), 256)
    # Gợi ý 2: a = cu.select(khoa, bam.alias("h_cu")); b = moi.select(khoa, bam.alias("h_moi"));
    #          full_outer join theo khoa; op = 'c' nếu h_cu null, 'd' nếu h_moi null, 'u' nếu khác nhau
    # Gợi ý 3: chỉ giữ dòng có op, rồi left join với `moi` để lấy giá trị mới
    ...
    return None


t0 = time.perf_counter()
thay_doi = phat_hien_thay_doi(snap1, snap2)
dem_op = None   # Gợi ý: {r["op"]: r["count"] for r in thay_doi.groupBy("op").count().collect()}
t_snapshot = time.perf_counter() - t0

In [ ]:
# Ô TỰ KIỂM TRA
assert dem_op == {k: len(SU_THAT_CDC[k]) for k in ["c", "u", "d"]}, f"số thay đổi chưa khớp: {dem_op}"
ids_u = {r["id"] for r in thay_doi.where("op = 'u'").select("id").collect()}
assert ids_u == SU_THAT_CDC["u"], "tập id bị sửa chưa khớp (kể cả sửa ngầm)"
assert thay_doi.where("op = 'd'").where(F.col("ma_sv").isNotNull()).count() == 0, "op='d' phải có giá trị mới null"
print("Ví dụ sự kiện thay đổi kiểu Debezium (rút gọn):")
for r in thay_doi.where("op = 'u'").limit(2).select(F.to_json(F.struct("op", F.struct("id", "diem_qt", "diem_thi", "diem_tk").alias("after"))).alias("j")).collect():
    print("  ", r["j"])
print(f"Thời gian: so sánh ảnh chụp {t_snapshot:.2f} s | nạp gia tăng {t_gia_tang:.2f} s")
dat("Bài tập 2.1")

### Bài tập 2.2 — Áp dụng thay đổi một cách lũy đẳng
Viết `ap_dung_thay_doi(dich, thay_doi, khoa)` trả về bảng đích sau khi áp dụng: bỏ các khóa có trong `thay_doi`, rồi thêm lại bản ghi mới của `op ∈ {c, u}`. Ô kiểm tra yêu cầu (i) kết quả trùng khớp ảnh chụp 2; (ii) áp dụng hai lần cho cùng kết quả.

In [ ]:
# BÀI TẬP 2.2 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def ap_dung_thay_doi(dich, thay_doi, khoa="id"):
    # Gợi ý: giu_lai = dich.join(thay_doi.select(khoa), khoa, "left_anti")
    #        moi = thay_doi.where(op thuộc {c,u}).drop("op").select(*dich.columns); return giu_lai.unionByName(moi)
    ...
    return None


diem_cdc = ap_dung_thay_doi(snap1, thay_doi)

In [ ]:
# Ô TỰ KIỂM TRA
def bang_nhau(a, b):
    return a.exceptAll(b).count() == 0 and b.exceptAll(a).count() == 0

assert diem_cdc is not None and bang_nhau(diem_cdc, snap2), "kết quả phải trùng khớp ảnh chụp 2"
assert bang_nhau(ap_dung_thay_doi(diem_cdc, thay_doi), snap2), "áp dụng lại phải cho cùng kết quả (lũy đẳng)"
print(f"Nạp gia tăng theo updated_at lệch {diem_gia_tang.exceptAll(snap2).count()} dòng so với nguồn; "
      f"áp dụng CDC lệch 0 dòng.")
dat("Bài tập 2.2")

### Câu hỏi thảo luận — Phần 2
1. (Câu hỏi thảo luận 1 của chương) Dựa trên số liệu ở Ví dụ 2b và Bài tập 2.1, giải thích vì sao CDC dựa trên nhật ký giao dịch thường được ưu tiên hơn truy vấn theo cột thời gian. Khi nào lựa chọn ngược lại hợp lý hơn (gợi ý: quyền truy cập binlog, bảng chỉ ghi nối, yêu cầu độ tươi theo ngày)?
2. So sánh ảnh chụp có độ phức tạp $O(N)$ trên cả hai ảnh chụp và cần xáo trộn theo khóa. Với bảng 500 triệu dòng thay đổi 0,1%/ngày, ước lượng khối lượng phải đọc của ba cách tiếp cận và đề xuất phương án.
3. Giao dịch dài có thể cam kết với `updated_at` nhỏ hơn dấu mốc đã lưu, khiến nạp gia tăng bỏ sót. Phân tích chiến lược "lùi dấu mốc một khoảng $\Delta$" kết hợp hợp nhất lũy đẳng: chọn $\Delta$ thế nào, chi phí ra sao?

---
## Phần 3. Hàng đợi kiểu Kafka: partition, consumer group, offset và ngữ nghĩa giao nhận

### Khái niệm từ bài giảng
*Slide mục 2.2: "Apache Kafka: nhật ký sự kiện làm trục tích hợp", "Kafka: consumer group và lưu giữ", "Kafka: ngữ nghĩa giao nhận"; mục 2.1: "Yêu cầu thiết kế tầng thu nhận"; Tài liệu đọc thêm: "Ước lượng năng lực tầng thu nhận".*

- **Topic** chia thành $P$ **partition**; mỗi partition là **nhật ký chỉ ghi nối** (append-only log), bản ghi có **offset** tăng dần. Cùng khóa vào cùng partition: $p = \mathrm{hash}(\text{key}) \bmod P$, nên thứ tự chỉ được bảo đảm **trong từng partition** (tức theo từng khóa).
- Trong một **consumer group**, mỗi partition được gán cho tối đa một consumer: mức song song tối đa bằng $P$. Các nhóm khác nhau đọc độc lập nhờ **lưu giữ** (retention).
- Consumer **xác nhận offset** (commit) sau khi ghi kết quả → **ít nhất một lần** (at-least-once): sự cố giữa "ghi" và "commit" gây xử lý lặp. Kết hợp **khóa sự kiện duy nhất** và đích **lũy đẳng** để đạt hiệu quả tương đương **đúng một lần**.
- **Độ trễ tiêu thụ** (consumer lag) $=$ offset cuối của partition $-$ offset đã commit.

Lớp Python dưới đây mô phỏng các cơ chế trên (không thay thế Kafka thật). Hàm băm dùng CRC32 thay cho murmur2 của Kafka; hàm `hash()` của Python không dùng được vì thay đổi giữa các tiến trình.

In [ ]:
class Topic:
    def __init__(self, ten, so_partition):
        self.ten, self.P = ten, so_partition
        self.log = [[] for _ in range(so_partition)]          # mỗi partition: nhật ký chỉ ghi nối

    def append(self, p, ban_ghi):
        offset = len(self.log[p])
        self.log[p].append({**ban_ghi, "partition": p, "offset": offset})
        return offset

    def end_offsets(self):
        return [len(l) for l in self.log]


def partition_theo_khoa(key, P):
    return zlib.crc32(key.encode("utf-8")) % P                 # Kafka dùng murmur2; ý tưởng như nhau


class Producer:
    def __init__(self, topic):
        self.topic = topic

    def send(self, key, value):
        p = partition_theo_khoa(key, self.topic.P)
        return p, self.topic.append(p, {"key": key, "value": value})


class ConsumerGroup:
    def __init__(self, topic, group_id):
        self.topic, self.group_id = topic, group_id
        self.committed = [0] * topic.P                         # offset kế tiếp cần đọc của từng partition

    def poll(self, p, max_records):
        bat_dau = self.committed[p]
        return self.topic.log[p][bat_dau: bat_dau + max_records]

    def commit(self, p, offset_ke_tiep):
        self.committed[p] = offset_ke_tiep

    def lag(self):
        return [e - c for e, c in zip(self.topic.end_offsets(), self.committed)]


# Producer gửi 30 000 sự kiện LMS (đã khử bản gửi lại từ nguồn), khóa = mã sinh viên
N_KAFKA = 30_000
su_kien = (lms_hop_le.dropDuplicates(["event_id"]).orderBy("thoi_gian", "event_id").limit(N_KAFKA)
           .select("event_id", "ma_sv", "loai", F.date_format("thoi_gian", "yyyy-MM-dd'T'HH:mm:ss").alias("thoi_gian"))
           .toPandas().to_dict("records"))
topic = Topic("lms.su_kien", so_partition=6)
prod = Producer(topic)
for ev in su_kien:
    prod.send(key=ev["ma_sv"], value=ev)
print("Số bản ghi mỗi partition:", topic.end_offsets())

# Thứ tự theo khóa: trong một partition, sự kiện của cùng sinh viên giữ đúng thứ tự gửi
dung_thu_tu = all(
    all(a <= b for a, b in zip(ts, ts[1:]))
    for p in range(topic.P)
    for ts in pd.DataFrame([r["value"] for r in topic.log[p]]).groupby("ma_sv")["thoi_gian"].apply(list))
print("Thứ tự theo từng sinh viên được bảo toàn:", dung_thu_tu)

# Hai consumer group đọc độc lập: nhóm ETL đã đọc 4 000 bản ghi mỗi partition, nhóm giám sát đọc hết
etl, giam_sat = ConsumerGroup(topic, "etl"), ConsumerGroup(topic, "giam_sat")
for p in range(topic.P):
    etl.commit(p, min(4000, topic.end_offsets()[p]))
    giam_sat.commit(p, topic.end_offsets()[p])
print("Lag nhóm etl     :", etl.lag())
print("Lag nhóm giam_sat:", giam_sat.lag())

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar([f"P{p}" for p in range(topic.P)], topic.end_offsets(), color="#2b5d8a")
ax.axhline(N_KAFKA / topic.P, ls="--", color="gray", label="phân bố đều")
ax.set_ylim(0, 1.3 * max(topic.end_offsets()))
ax.set_ylabel("Số bản ghi"); ax.set_title("Phân bố bản ghi theo partition (khóa = mã SV)"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

### Bài tập 3.1 — Phân công partition trong consumer group (range assignor)
Viết `phan_cong_range(P, consumers)` trả về `dict` consumer → danh sách partition liên tiếp; $q = \lfloor P/n \rfloor$, $r = P \bmod n$, $r$ consumer đầu nhận $q+1$ partition, các consumer còn lại nhận $q$.

In [ ]:
# BÀI TẬP 3.1 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def phan_cong_range(P, consumers):
    n = len(consumers)
    q, r = divmod(P, n)
    ket_qua, bat_dau = {}, 0
    # Gợi ý: duyệt consumers; consumer thứ i nhận q + (1 if i < r else 0) partition liên tiếp từ bat_dau
    ...
    return ket_qua

In [ ]:
# Ô TỰ KIỂM TRA
pc = phan_cong_range(6, ["c1", "c2", "c3", "c4"])
assert pc == {"c1": [0, 1], "c2": [2, 3], "c3": [4], "c4": [5]}, pc
pc8 = phan_cong_range(6, [f"c{i}" for i in range(1, 9)])
assert sorted(p for v in pc8.values() for p in v) == list(range(6)), "mỗi partition được gán đúng một lần"
print("6 partition, 8 consumer:", pc8, "→", sum(1 for v in pc8.values() if not v), "consumer rảnh")
dat("Bài tập 3.1")

### Bài tập 3.2 — Ít nhất một lần, sự cố trước khi commit và đích lũy đẳng
1. Hoàn thiện `chay_consumer`: đọc partition `p` theo lô; với mỗi lô, gọi `ghi(rec)` cho từng bản ghi **rồi mới** commit offset kế tiếp. Nếu `su_co_tai_lo` bằng chỉ số lô hiện tại thì phát sinh `SuCoConsumer` **sau khi ghi, trước khi commit**.
2. Viết `ghi_luy_dang(rec)`: ghi vào `dich_luy_dang` (dict) theo khóa `event_id` (upsert).

Ô kiểm tra chạy kịch bản: consumer partition 0 gặp sự cố ở lô thứ 2, khởi động lại và đọc tiếp từ offset đã commit.

In [ ]:
# BÀI TẬP 3.2 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
class SuCoConsumer(Exception):
    pass


def chay_consumer(nhom, p, ghi, kich_thuoc_lo=500, su_co_tai_lo=None):
    lo = 0
    while True:
        batch = nhom.poll(p, kich_thuoc_lo)
        if not batch:
            return
        # Gợi ý: (1) gọi ghi(rec) cho mọi rec trong batch
        #        (2) nếu su_co_tai_lo == lo: raise SuCoConsumer(...)
        #        (3) nhom.commit(p, batch[-1]["offset"] + 1); lo += 1
        ...
        return


dich_noi, dich_luy_dang = [], {}


def ghi_noi(rec):                     # đích chỉ ghi nối: KHÔNG lũy đẳng
    dich_noi.append(rec["value"]["event_id"])


def ghi_luy_dang(rec):                # đích lũy đẳng: upsert theo khóa sự kiện
    ...

In [ ]:
# Ô TỰ KIỂM TRA
nhom_kho = ConsumerGroup(topic, "kho_du_lieu")
ghi_ca_hai = lambda rec: (ghi_noi(rec), ghi_luy_dang(rec))
so_su_co = 0
for p in range(topic.P):
    try:
        chay_consumer(nhom_kho, p, ghi_ca_hai, kich_thuoc_lo=500, su_co_tai_lo=1 if p == 0 else None)
    except SuCoConsumer as e:
        so_su_co += 1
        print("Sự cố:", e, "→ khởi động lại consumer")
        chay_consumer(nhom_kho, p, ghi_ca_hai, kich_thuoc_lo=500)
assert so_su_co == 1, "kịch bản phải phát sinh đúng một sự cố"
assert nhom_kho.lag() == [0] * topic.P, "sau khi khởi động lại phải đọc hết (lag = 0)"
assert len(dich_noi) == N_KAFKA + 500, f"đích ghi nối phải có 500 bản ghi lặp, hiện có {len(dich_noi) - N_KAFKA}"
assert len(dich_luy_dang) == N_KAFKA, "đích lũy đẳng phải có đúng N_KAFKA sự kiện"
print(f"Đích ghi nối: {len(dich_noi)} bản ghi ({len(dich_noi) - len(set(dich_noi))} bản lặp) | đích lũy đẳng: {len(dich_luy_dang)}")
dat("Bài tập 3.2")

### Bài tập 3.3 — Ước lượng số partition và dung lượng lưu giữ
Theo Tài liệu đọc thêm Chương 2: $P \ge \max(R/r_p,\; R/r_c)$ và $C = R \times t_{\text{giữ}} \times f_{\text{sao chép}}$.
Viết `uoc_luong_partition(R, r_p, r_c)` (làm tròn lên) và `dung_luong_luu_giu_TB(R_MBps, so_ngay, he_so_sao_chep)` (đơn vị TB thập phân, 1 TB = $10^6$ MB). Sau đó áp dụng cho LMS với các tham số **giả định** trong ô.

In [ ]:
# BÀI TẬP 3.3 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def uoc_luong_partition(R, r_p, r_c):
    # Gợi ý: math.ceil(max(R / r_p, R / r_c))
    return None


def dung_luong_luu_giu_TB(R_MBps, so_ngay, he_so_sao_chep):
    # Gợi ý: R_MBps * 86_400 * so_ngay * he_so_sao_chep / 1e6
    return None


# Áp dụng cho LMS — tham số GIẢ ĐỊNH: đỉnh 3 000 sự kiện/s, 250 byte/sự kiện, r_p = 5 MB/s, r_c = 0,25 MB/s
R_lms = 3000 * 250 / 1e6
P_lms = ...
C_lms = ...

In [ ]:
# Ô TỰ KIỂM TRA
assert uoc_luong_partition(200, 20, 10) == 20, "ví dụ trên slide: P ≥ 20"
assert abs(dung_luong_luu_giu_TB(200, 3, 3) - 155.52) < 1e-6, "ví dụ trên slide: ≈ 155 TB"
assert P_lms == 3 and abs(C_lms - 1.3608) < 1e-6
dat("Bài tập 3.3")

### Câu hỏi thảo luận — Phần 3
1. Khóa phân vùng là mã sinh viên: thứ tự được bảo đảm theo sinh viên. Nếu một tài khoản tự động sinh 30% sự kiện (khóa "nóng"), điều gì xảy ra với partition và consumer tương ứng? Đề xuất cách xử lý và hệ quả đối với thứ tự.
2. Tăng $P$ từ 6 lên 12 làm thay đổi $\mathrm{hash}(\text{key}) \bmod P$. Phân tích ảnh hưởng tới thứ tự theo khóa và tới các bộ xử lý có trạng thái phía sau.
3. Ở Bài tập 3.2, đích lũy đẳng giải quyết trùng lặp do consumer. Trùng lặp do producer gửi lại (2% trong nhật ký LMS) được xử lý ở đâu? So sánh `enable.idempotence=true` phía producer với khử trùng lặp ở đích.

---
## Phần 4. Khám phá dữ liệu (EDA) ở quy mô lớn

### Khái niệm từ bài giảng
*Slide mục 2.1: "Khám phá dữ liệu (EDA) ở quy mô lớn", "Các bước khám phá dữ liệu", "Thống kê mô tả và phân vị xấp xỉ với PySpark", "Lấy mẫu để khám phá dữ liệu lớn", "Giảm dữ liệu trước khi trực quan hóa".*

- `describe()` chỉ cần $n, \sum x, \sum x^2$: một lượt quét song song.
- `approxQuantile(col, probs, ε)` (biến thể Greenwald–Khanna): hạng của phân vị trả về lệch không quá $\varepsilon N$; $\varepsilon = 0$ cho kết quả chính xác nhưng tốn kém.
- `approx_count_distinct(col, rsd)` dùng HyperLogLog++: bộ nhớ cố định, sai số chuẩn tương đối khoảng `rsd`; `countDistinct` chính xác nhưng cần xáo trộn.
- Ngoại lai theo **IQR**: ngoài khoảng $[Q_1 - 1{,}5\,\mathrm{IQR},\; Q_3 + 1{,}5\,\mathrm{IQR}]$.
- **Lấy mẫu phân tầng** `sampleBy(col, fractions)` giữ nhóm nhỏ nhưng quan trọng; sai số chuẩn $\mathrm{SE}(\bar x) = \sigma/\sqrt{n}$ hầu như không phụ thuộc cỡ tổng thể $N$ khi $n \ll N$.

### Ví dụ mẫu 4 — Hồ sơ dữ liệu, phân vị xấp xỉ và đếm phân biệt xấp xỉ

In [ ]:
lms = lms_hop_le
n_lms = lms.count()
print("Số sự kiện:", n_lms)
lms.select("thoi_luong_giay", "diem_quiz").describe().show()

# --- approxQuantile: thời gian và sai lệch hạng thực nghiệm |F(q) - p| so với cận ε ---
PROBS = [0.25, 0.5, 0.75, 0.99]
co_gia_tri = lms.where(F.col("thoi_luong_giay").isNotNull()).cache(); co_gia_tri.count()
dong = []
for eps in [0.1, 0.01, 0.001, 0.0]:
    t0 = time.perf_counter()
    q = lms.approxQuantile("thoi_luong_giay", PROBS, eps)
    dt = time.perf_counter() - t0
    F_q = co_gia_tri.select(*[F.avg((F.col("thoi_luong_giay") <= v).cast("double")).alias(f"p{i}") for i, v in enumerate(q)]).first()
    dong.append({"ε": eps, "Q1": q[0], "trung vị": q[1], "Q3": q[2], "P99": q[3],
                 "sai lệch hạng lớn nhất": round(max(abs(F_q[i] - p) for i, p in enumerate(PROBS)), 4),
                 "thời gian (s)": round(dt, 2)})
print(pd.DataFrame(dong).to_string(index=False))

# --- countDistinct so với approx_count_distinct trên một bảng tổng hợp lớn hơn ---
N_LON = 3_000_000
lon = spark.range(N_LON).select((F.abs(F.xxhash64("id")) % 1_500_000).alias("k")).cache(); lon.count()
t0 = time.perf_counter(); chinh_xac = lon.agg(F.countDistinct("k")).first()[0]; t_cx = time.perf_counter() - t0
dong = [{"phương pháp": "countDistinct", "rsd": "-", "kết quả": chinh_xac, "sai số tương đối (%)": 0.0, "thời gian (s)": round(t_cx, 2)}]
for rsd in [0.05, 0.02, 0.01]:
    t0 = time.perf_counter(); xx = lon.agg(F.approx_count_distinct("k", rsd=rsd)).first()[0]; dt = time.perf_counter() - t0
    dong.append({"phương pháp": "approx_count_distinct", "rsd": rsd, "kết quả": xx,
                 "sai số tương đối (%)": round(100 * abs(xx - chinh_xac) / chinh_xac, 2), "thời gian (s)": round(dt, 2)})
print(pd.DataFrame(dong).to_string(index=False))

# --- Giảm dữ liệu trước khi vẽ: tính histogram trong Spark, chỉ đưa ~40 cột về driver ---
BIN = 0.1
hist = (co_gia_tri.where("thoi_luong_giay > 0")
        .groupBy(F.floor(F.log10("thoi_luong_giay") / BIN).alias("b")).count().orderBy("b").toPandas())
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(hist["b"] * BIN, hist["count"], width=BIN, align="edge", color="#2b5d8a")
ax.set_xlabel("log10(thời lượng phiên, giây)"); ax.set_ylabel("Số sự kiện")
ax.set_title("Phân phối thời lượng phiên LMS (dữ liệu mô phỏng) — lệch phải, có đuôi dài")
plt.tight_layout(); plt.show()

### Bài tập 4.1 — Tỷ lệ thiếu theo cột trong **một** lượt quét
Viết `ty_le_thieu(df)` trả về `dict` cột → tỷ lệ null, dùng **một** phép `select` (gộp cả đếm tổng số dòng).

In [ ]:
# BÀI TẬP 4.1 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def ty_le_thieu(df):
    # Gợi ý: row = df.select(F.count(F.lit(1)).alias("__n"), *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first()
    ...
    return None


thieu = ty_le_thieu(lms)

In [ ]:
# Ô TỰ KIỂM TRA
assert isinstance(thieu, dict) and set(thieu) == set(lms.columns)
for c in ["thoi_luong_giay", "diem_quiz", "ma_hp"]:
    assert abs(thieu[c] - lms.where(F.col(c).isNull()).count() / n_lms) < 1e-12, c
assert abs((1 - thieu["thoi_luong_giay"]) - SU_THAT_SINH["ty_le_day_du_thoi_luong"]) < 1e-9
dat("Bài tập 4.1")

### Bài tập 4.2 — Ngoại lai theo IQR với phân vị xấp xỉ
Viết `bien_iqr(df, cot, eps)` trả về `(q1, q3, duoi, tren)`; tính `so_ngoai_lai` của `thoi_luong_giay` với `eps = 0.001`. Ô kiểm tra so sánh với phân vị chính xác và vẽ biên trên histogram.

In [ ]:
# BÀI TẬP 4.2 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def bien_iqr(df, cot, eps=0.001):
    # Gợi ý: q1, q3 = df.approxQuantile(cot, [0.25, 0.75], eps); iqr = q3 - q1
    ...
    return None


q1, q3, duoi, tren = None, None, None, None   # Gợi ý: = bien_iqr(lms, "thoi_luong_giay", 0.001)
so_ngoai_lai = None

In [ ]:
# Ô TỰ KIỂM TRA
q1x, q3x, duoix, trenx = bien_iqr(lms, "thoi_luong_giay", 0.0)
assert abs(tren - trenx) / trenx < 0.03, "biên trên xấp xỉ phải gần biên chính xác"
n_ngoai_lai_x = lms.where((F.col("thoi_luong_giay") < duoix) | (F.col("thoi_luong_giay") > trenx)).count()
assert abs(so_ngoai_lai - n_ngoai_lai_x) / n_ngoai_lai_x < 0.1
assert lms.where((F.col("thoi_luong_giay") >= 20_000) & (F.col("thoi_luong_giay") <= tren)).count() == 0, "phiên bỏ quên phải bị đánh dấu"
print(f"Số ngoại lai: xấp xỉ {so_ngoai_lai} | chính xác {n_ngoai_lai_x}")
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(hist["b"] * BIN, hist["count"], width=BIN, align="edge", color="#9ecae1")
ax.axvline(np.log10(tren), color="#b30000", ls="--", label=f"biên trên IQR = {tren:.0f} s")
ax.set_xlabel("log10(thời lượng phiên, giây)"); ax.set_ylabel("Số sự kiện"); ax.legend()
ax.set_title("Biên ngoại lai theo IQR"); plt.tight_layout(); plt.show()
dat("Bài tập 4.2")

### Bài tập 4.3 — Lấy mẫu phân tầng theo khoa
Các khoa có quy mô rất khác nhau. Hãy lấy mẫu sao cho **mỗi khoa khoảng 2 000 sự kiện** bằng `sampleBy("ma_khoa", ti_le, seed=SEED)`, rồi lấy một mẫu ngẫu nhiên đơn giản có cùng cỡ để so sánh.

In [ ]:
lms_khoa = lms.join(sv_raw.select("ma_sv", "ma_khoa"), "ma_sv").cache()
dem_khoa = {r["ma_khoa"]: r["count"] for r in lms_khoa.groupBy("ma_khoa").count().collect()}
print("Số sự kiện theo khoa:", dict(sorted(dem_khoa.items(), key=lambda kv: -kv[1])))

In [ ]:
# BÀI TẬP 4.3 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
MUC_TIEU = 2000
ti_le = ...      # Gợi ý: {k: min(1.0, MUC_TIEU / v) for k, v in dem_khoa.items()}
mau_tang = ...   # Gợi ý: lms_khoa.sampleBy("ma_khoa", fractions=ti_le, seed=SEED)
mau_don = ...    # Gợi ý: lms_khoa.sample(fraction=<cỡ mẫu tầng / tổng>, seed=SEED)

In [ ]:
# Ô TỰ KIỂM TRA
c_tang = {r["ma_khoa"]: r["count"] for r in mau_tang.groupBy("ma_khoa").count().collect()}
c_don = {r["ma_khoa"]: r["count"] for r in mau_don.groupBy("ma_khoa").count().collect()}
assert all(0.85 * MUC_TIEU <= c_tang[k] <= 1.15 * MUC_TIEU for k in dem_khoa), c_tang
tong_the = lms_khoa.where("loai = 'lam_quiz'").groupBy("ma_khoa").agg(F.avg("diem_quiz").alias("tong_the")).toPandas().set_index("ma_khoa")
ss = pd.DataFrame({"tổng thể": pd.Series(dem_khoa), "mẫu phân tầng": pd.Series(c_tang), "mẫu đơn giản": pd.Series(c_don)})
for ten, mau in [("TB quiz – phân tầng", mau_tang), ("TB quiz – đơn giản", mau_don)]:
    ss[ten] = mau.where("loai = 'lam_quiz'").groupBy("ma_khoa").agg(F.avg("diem_quiz")).toPandas().set_index("ma_khoa").iloc[:, 0]
ss["TB quiz – tổng thể"] = tong_the["tong_the"]
print(ss.sort_values("tổng thể", ascending=False).round(3).to_string())
khoa_nho = min(dem_khoa, key=dem_khoa.get)
assert c_don.get(khoa_nho, 0) < 0.5 * c_tang[khoa_nho], "mẫu đơn giản phải có ít quan sát của khoa nhỏ hơn hẳn"
dat("Bài tập 4.3")

### Câu hỏi thảo luận — Phần 4
1. Từ bảng `approxQuantile`, sai lệch hạng thực nghiệm có luôn nằm dưới $\varepsilon$ không? Vì sao dữ liệu có nhiều giá trị trùng (thời lượng tính theo giây nguyên) làm sai lệch hạng khó diễn giải?
2. HyperLogLog cho phép **gộp** (merge) bản phác thảo của từng ngày để có số sinh viên hoạt động trong tuần mà không đọc lại dữ liệu. Vì sao `countDistinct` không có tính chất này? Khi nào **không** được dùng đếm xấp xỉ (ví dụ số liệu báo cáo theo quy định)?
3. Kết quả tính trên mẫu phân tầng phải được **gia quyền** thế nào để ước lượng trung bình toàn trường? Viết công thức và giải thích sai lệch nếu bỏ qua trọng số.

---
## Phần 5. Làm sạch và tích hợp dữ liệu

### Khái niệm từ bài giảng
*Slide mục 2.2: "Sáu chiều chất lượng dữ liệu và cách đo", "Khử trùng lặp ở quy mô lớn", "Đối sánh thực thể (entity resolution)", "Đối sánh thực thể: độ phức tạp và phân khối", "Cơ chế thiếu dữ liệu"; mục 2.1: "Tiền xử lý dữ liệu", "Rò rỉ dữ liệu trong tiền xử lý"; Tài liệu đọc thêm: "Mô hình chuẩn và chuẩn hóa Unicode".*

- Tiếng Việt có hai dạng biểu diễn Unicode thường gặp: **dựng sẵn** (NFC, "ễ" là một ký tự) và **tổ hợp** (NFD, "e" + dấu mũ + dấu ngã). Hai chuỗi nhìn giống nhau nhưng khác nhau khi so sánh, nối bảng, đếm phân biệt → phải chuẩn hóa NFC trước mọi phép so khớp.
- **Khử trùng lặp**: trùng lặp chính xác theo khóa sự kiện (`dropDuplicates`); nhiều phiên bản theo khóa nghiệp vụ (hàm cửa sổ giữ bản mới nhất); trùng lặp gần đúng cần **đối sánh thực thể**.
- **Đối sánh thực thể**: chuẩn hóa → **phân khối** (blocking) → so sánh cặp trong khối → phân lớp khớp/không → gom cụm. So sánh mọi cặp cần $\binom{N}{2} \approx N^2/2$ phép; phân khối thành $B$ khối còn khoảng $N^2/(2B)$. Độ tương tự Jaccard trên tập 2-gram: $J(A, B) = |A \cap B| / |A \cup B|$.
- **Cơ chế thiếu dữ liệu** (Rubin, 1976): MCAR, MAR, MNAR. Điền giá trị thiếu bằng `Imputer` (trung bình/trung vị); khi dùng cho học máy chỉ `fit` trên tập huấn luyện để tránh rò rỉ dữ liệu.
- **Sáu chiều chất lượng**: đầy đủ, chính xác, nhất quán, kịp thời, duy nhất, hợp lệ — mỗi chiều cần một độ đo cụ thể và ngưỡng chấp nhận.

### Ví dụ mẫu 5a — Unicode NFC/NFD và chuẩn hóa họ tên

In [ ]:
a = "Nguyễn"; b = unicodedata.normalize("NFD", a)
print(f"'{a}' == '{b}' ? {a == b} | độ dài {len(a)} và {len(b)} | mã ký tự NFD:", [hex(ord(ch)) for ch in b])

nfc = F.udf(lambda s: None if s is None else unicodedata.normalize("NFC", s), T.StringType())
la_nfc = F.udf(lambda s: s is None or unicodedata.is_normalized("NFC", s), T.BooleanType())


def chuan_hoa_ten_col(c):
    """NFC -> gộp khoảng trắng -> bỏ khoảng trắng đầu/cuối -> viết hoa chữ cái đầu mỗi từ."""
    return F.initcap(F.trim(F.regexp_replace(nfc(c), r"\s+", " ")))


sv_raw.select(F.sum((~la_nfc("ho_ten")).cast("int")).alias("không phải NFC"),
              F.sum(F.col("ho_ten").rlike(r"^\s|\s$|\s{2,}").cast("int")).alias("thừa khoảng trắng"),
              F.sum((F.col("ho_ten") == F.upper("ho_ten")).cast("int")).alias("viết hoa toàn bộ")).show()
(sv_raw.select("ho_ten", chuan_hoa_ten_col(F.col("ho_ten")).alias("ho_ten_chuan"))
 .where(F.col("ho_ten") != F.col("ho_ten_chuan")).show(6, truncate=False))

# Các định dạng ngày sinh và số điện thoại đang tồn tại trong nguồn
dang_ngay = (F.when(F.col("ngay_sinh").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
              .when(F.col("ngay_sinh").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
              .when(F.col("ngay_sinh").rlike(r"^\d{2}-\d{2}-\d{4}$"), "dd-MM-yyyy")
              .when(F.col("ngay_sinh").rlike(r"^\d{8}$"), "yyyyMMdd").otherwise("null/khác"))
sv_raw.groupBy(dang_ngay.alias("định dạng ngày sinh")).count().orderBy(F.desc("count")).show()
sv_raw.select("so_dt").where(F.col("so_dt").isNotNull()).distinct().limit(6).show()

### Ví dụ mẫu 5b — Khử trùng lặp sự kiện và điền giá trị thiếu

In [ ]:
# (1) Trùng lặp chính xác theo khóa sự kiện; giữ bản ghi đến SỚM nhất bằng hàm cửa sổ (xác định được)
w_ev = Window.partitionBy("event_id").orderBy("thoi_gian_nap")
lms_dd = lms.withColumn("rn", F.row_number().over(w_ev)).where("rn = 1").drop("rn").cache()
print(f"Sự kiện trước/sau khử trùng lặp: {n_lms} → {lms_dd.count()} (đã bỏ {n_lms - lms_dd.count()} bản gửi lại)")

# (2) Giá trị thiếu: tỷ lệ thiếu điểm quá trình theo khoa (dấu hiệu MAR)
diem_khoa = diem_raw.join(sv_raw.select("ma_sv", "ma_khoa"), "ma_sv", "left").cache()
(diem_khoa.groupBy("ma_khoa").agg(F.round(F.avg(F.col("diem_qt").isNull().cast("double")), 3).alias("tỷ lệ thiếu diem_qt"),
                                  F.count("*").alias("số dòng"))
 .orderBy(F.desc("tỷ lệ thiếu diem_qt")).show())

# (3) Imputer (trung vị toàn cục) + cột chỉ báo thiếu; so sánh với điền trung vị theo khoa
from pyspark.ml.feature import Imputer
mo_hinh_dien = Imputer(inputCols=["diem_qt"], outputCols=["diem_qt_dien"], strategy="median").fit(diem_khoa)
diem_dien = (mo_hinh_dien.transform(diem_khoa)
             .withColumn("diem_qt_thieu", F.col("diem_qt").isNull().cast("int"))
             .join(diem_khoa.groupBy("ma_khoa").agg(F.percentile_approx("diem_qt", 0.5).alias("trung_vi_khoa")), "ma_khoa", "left")
             .withColumn("diem_qt_dien_theo_khoa", F.coalesce("diem_qt", "trung_vi_khoa")))
print("Trung vị toàn cục dùng để điền:", mo_hinh_dien.surrogateDF.first()[0])
(diem_dien.where("diem_qt_thieu = 1").groupBy("ma_khoa")
 .agg(F.first("diem_qt_dien").alias("điền toàn cục"), F.first("diem_qt_dien_theo_khoa").alias("điền theo khoa"))
 .orderBy("ma_khoa").show())

### Bài tập 5.1 — Chuẩn hóa hồ sơ sinh viên
Viết `chuan_hoa_sinh_vien(df)` trả về DataFrame cùng số dòng, thay các cột:
- `ho_ten`: dùng `chuan_hoa_ten_col`;
- `ngay_sinh`: kiểu `DateType`, thử lần lượt các định dạng trong `DINH_DANG_NGAY`;
- `so_dt`: chỉ giữ chữ số, đổi tiền tố `84` (11 chữ số) thành `0`, hợp lệ khi khớp `^0[35789][0-9]{8}$`, không hợp lệ thì để null;
- `email`: chữ thường, bỏ khoảng trắng.

Ô kiểm tra đối chiếu với **hồ sơ gốc đã xác minh** (`tham_chieu_ho_so.csv`).

In [ ]:
# BÀI TẬP 5.1 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
DINH_DANG_NGAY = ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy", "yyyyMMdd"]


def chuan_hoa_sinh_vien(df):
    # Gợi ý ngày sinh: F.coalesce(*[F.to_date(F.col("ngay_sinh"), f) for f in DINH_DANG_NGAY])
    # Gợi ý số điện thoại: so = F.regexp_replace(F.col("so_dt"), r"\D", "");
    #   so = F.when(so.startswith("84") & (F.length(so) == 11), F.concat(F.lit("0"), so.substr(3, 9))).otherwise(so)
    #   F.when(so.rlike(r"^0[35789][0-9]{8}$"), so)
    ...
    return None


sv_chuan = chuan_hoa_sinh_vien(sv_raw)

In [ ]:
# Ô TỰ KIỂM TRA
tc = spark.read.csv(f"{DATA_DIR}/nguon/tham_chieu_ho_so.csv", header=True,
                    schema="ma_sv string, ho_ten_tc string, ngay_sinh_tc date, so_dt_tc string")
assert sv_chuan.count() == sv_raw.count()
assert dict(sv_chuan.dtypes)["ngay_sinh"] == "date"
bad = sv_chuan.where((~la_nfc("ho_ten")) | F.col("ho_ten").rlike(r"^\s|\s$|\s{2,}")).count()
assert bad == 0, "họ tên phải ở dạng NFC và không thừa khoảng trắng"
j = tc.join(sv_chuan, "ma_sv")
kq = j.select(F.avg((F.col("ho_ten") == F.col("ho_ten_tc")).cast("double")).alias("ten"),
              F.avg(F.coalesce(F.col("ngay_sinh") == F.col("ngay_sinh_tc"), F.lit(False)).cast("double")).alias("ngay"),
              F.avg(F.when(F.col("so_dt").isNotNull(), (F.col("so_dt") == F.col("so_dt_tc")).cast("double"))).alias("sdt_khac_null")).first()
print(f"Khớp tham chiếu — họ tên: {kq['ten']:.3f} | ngày sinh: {kq['ngay']:.3f} | số điện thoại (khi khác null): {kq['sdt_khac_null']:.3f}")
assert kq["ten"] == 1.0 and kq["ngay"] >= 0.97 and kq["sdt_khac_null"] == 1.0
assert sv_chuan.where(F.col("email").rlike("[A-Z]")).count() == 0
dat("Bài tập 5.1")

### Bài tập 5.2 — Đối sánh thực thể với phân khối nhiều lượt
Bảng `sinh_vien` chứa khoảng 3% hồ sơ nhập trùng với **mã sinh viên khác** (tên không dấu, thiếu tên đệm, gõ sai, thiếu ngày sinh…). Hãy viết `doi_sanh_thuc_the(sv)` trả về `(cap_khop, so_cap_ung_vien)`:
1. tạo cột `ten` = `khoa_ten(ho_ten)` (bỏ dấu, chữ thường);
2. **phân khối hai lượt**: (`ngay_sinh`, `ma_khoa`) và (`so_dt`); tự nối trong khối với điều kiện `ma_sv_a < ma_sv_b`; hợp và bỏ trùng các cặp ứng viên;
3. tính `jaccard_2gram(ten_a, ten_b)`; phân lớp **khớp** nếu điểm ≥ `nguong`, hoặc cùng số điện thoại và điểm ≥ 0,5.

Ô kiểm tra tính **độ chuẩn xác** (precision), **độ phủ** (recall) so với khóa đáp án mô phỏng và **tỷ lệ giảm** số cặp so với so sánh mọi cặp.

In [ ]:
# Công cụ hỗ trợ (dùng chung)
def _bigram(s):
    s = f" {s} "
    return {s[i:i + 2] for i in range(len(s) - 1)}


@F.udf(T.DoubleType())
def jaccard_2gram(a, b):
    if a is None or b is None:
        return None
    A, B = _bigram(a), _bigram(b)
    return len(A & B) / len(A | B)


@F.udf(T.StringType())
def khoa_ten(s):
    return None if s is None else " ".join(bo_dau(unicodedata.normalize("NFC", s)).lower().split())


print(round(jaccard_2gram.func("nguyen van an", "nguyen an"), 3), round(jaccard_2gram.func("nguyen van an", "tran thi lan"), 3))

In [ ]:
# BÀI TẬP 5.2 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def doi_sanh_thuc_the(sv, nguong=0.7):
    k = sv.select("ma_sv", "ngay_sinh", "ma_khoa", "so_dt", khoa_ten("ho_ten").alias("ten"))
    a = k.select(*[F.col(c).alias(c + "_a") for c in k.columns])
    b = k.select(*[F.col(c).alias(c + "_b") for c in k.columns])
    # Gợi ý: def tu_noi(cot_khoi): điều kiện = bằng nhau trên mọi cột khối + F.col("ma_sv_a") < F.col("ma_sv_b")
    # ung_vien = tu_noi(["ngay_sinh", "ma_khoa"]).unionByName(tu_noi(["so_dt"])).dropDuplicates([...]) + cột "diem"
    # khop: diem >= nguong HOẶC (cùng so_dt VÀ diem >= 0.5); chú ý so sánh với null cho kết quả null
    ...
    return None, None


cap_khop, so_cap_ung_vien = doi_sanh_thuc_the(sv_chuan)

In [ ]:
# Ô TỰ KIỂM TRA
du_doan = {tuple(sorted((r["ma_a"], r["ma_b"]))) for r in cap_khop.collect()}
tp = len(du_doan & GT_CAP_TRUNG)
precision, recall = tp / max(1, len(du_doan)), tp / len(GT_CAP_TRUNG)
N_ = sv_chuan.count()
giam = 1 - so_cap_ung_vien / (N_ * (N_ - 1) / 2)
print(f"Độ chuẩn xác = {precision:.3f} | độ phủ = {recall:.3f} | cặp ứng viên = {so_cap_ung_vien} "
      f"(so với {N_ * (N_ - 1) // 2:,} cặp nếu so mọi cặp; giảm {100 * giam:.3f}%)")
assert precision >= 0.9 and recall >= 0.9, "cần độ chuẩn xác và độ phủ ≥ 0,9"
assert giam > 0.99
dat("Bài tập 5.2")

### Bài tập 5.3 — Bảng điểm chất lượng dữ liệu theo sáu chiều
Hoàn thiện `tinh_chi_so_chat_luong()` trả về `dict` sáu chỉ số trong $[0, 1]$:

| Khóa | Chiều | Định nghĩa trên dữ liệu mô phỏng |
|---|---|---|
| `day_du` | đầy đủ | tỷ lệ sự kiện trong `lms_hop_le` có `thoi_luong_giay` khác null |
| `chinh_xac` | chính xác | tỷ lệ hồ sơ tham chiếu có `so_dt` (sau chuẩn hóa, `sv_chuan`) trùng số điện thoại tham chiếu |
| `nhat_quan` | nhất quán | tỷ lệ dòng `diem_raw` có `ma_sv` tồn tại trong `sv_raw` |
| `kip_thoi` | kịp thời | tỷ lệ sự kiện có `thoi_gian_nap − thoi_gian ≤ 900` giây (SLA 15 phút) |
| `duy_nhat` | duy nhất | số `event_id` phân biệt / số sự kiện |
| `hop_le` | hợp lệ | tỷ lệ dòng `diem_raw` có `diem_thi` trong $[0, 10]$ và `diem_qt`, `diem_tk` (nếu khác null) trong $[0, 10]$ |

In [ ]:
# BÀI TẬP 5.3 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def tinh_chi_so_chat_luong():
    cs = {}
    # (1) Đầy đủ — ví dụ đã làm sẵn:
    cs["day_du"] = lms_hop_le.select(F.avg(F.col("thoi_luong_giay").isNotNull().cast("double"))).first()[0]
    # (2) Chính xác: đọc tham_chieu_ho_so.csv, left join sv_chuan theo ma_sv, so sánh so_dt (null tính là sai)
    cs["chinh_xac"] = ...
    # (3) Nhất quán: tỷ lệ dòng diem_raw có ma_sv xuất hiện trong sv_raw (gợi ý: left join với danh sách ma_sv)
    cs["nhat_quan"] = ...
    # (4) Kịp thời: F.unix_timestamp("thoi_gian_nap") - F.unix_timestamp("thoi_gian") <= 900
    cs["kip_thoi"] = ...
    # (5) Duy nhất: số event_id phân biệt / số dòng
    cs["duy_nhat"] = ...
    # (6) Hợp lệ: F.col(c).between(0, 10); diem_qt, diem_tk null được coi là hợp lệ
    cs["hop_le"] = ...
    return cs


chi_so = tinh_chi_so_chat_luong()

In [ ]:
# Ô TỰ KIỂM TRA
s = SU_THAT_SINH
ky_vong = {"day_du": s["ty_le_day_du_thoi_luong"], "chinh_xac": s["ty_le_chinh_xac_sdt"],
           "nhat_quan": 1 - s["n_diem_mo_coi"] / s["n_diem"], "kip_thoi": s["ty_le_kip_thoi_15p"],
           "duy_nhat": 1 - s["n_lms_trung"] / s["n_lms_hop_le"], "hop_le": 1 - s["n_diem_ngoai_mien"] / s["n_diem"]}
for k, v in ky_vong.items():
    assert chi_so.get(k) is not None and abs(chi_so[k] - v) < 1e-6, f"{k}: {chi_so.get(k)} ≠ {v}"
NGUONG = {"day_du": 0.95, "chinh_xac": 0.95, "nhat_quan": 0.995, "kip_thoi": 0.95, "duy_nhat": 0.99, "hop_le": 0.995}   # ngưỡng giả định
TEN_CHIEU = {"day_du": "Đầy đủ", "chinh_xac": "Chính xác", "nhat_quan": "Nhất quán", "kip_thoi": "Kịp thời",
             "duy_nhat": "Duy nhất", "hop_le": "Hợp lệ"}
bang_cl = pd.DataFrame([{"Chiều": TEN_CHIEU[k], "Giá trị": round(chi_so[k], 4), "Ngưỡng": NGUONG[k],
                         "Đánh giá": "đạt" if chi_so[k] >= NGUONG[k] else "KHÔNG đạt"} for k in NGUONG])
print(bang_cl.to_string(index=False))
fig, ax = plt.subplots(figsize=(7.5, 3.6))
mau = ["#41ab5d" if r["Đánh giá"] == "đạt" else "#cb181d" for _, r in bang_cl.iterrows()]
ax.barh(bang_cl["Chiều"], bang_cl["Giá trị"], color=mau)
ax.scatter(bang_cl["Ngưỡng"], bang_cl["Chiều"], marker="|", s=400, color="black", label="ngưỡng")
ax.set_xlim(0.85, 1.0); ax.set_xlabel("Giá trị chỉ số"); ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_title("Bảng điểm chất lượng dữ liệu (dữ liệu mô phỏng)"); plt.tight_layout(); plt.show()
dat("Bài tập 5.3")

### Câu hỏi thảo luận — Phần 5
1. Chuẩn hóa NFC không đồng nhất được "Hòa" và "Hoà" (khác vị trí đặt dấu thanh). Đề xuất quy tắc chuẩn hóa bổ sung và đánh giá rủi ro khi áp dụng tự động trên toàn bộ danh mục.
2. Ở Bài tập 5.2, nếu chỉ phân khối theo (`ngay_sinh`, `ma_khoa`) thì độ phủ thay đổi thế nào? Phân tích đánh đổi độ phủ – số cặp ứng viên – dữ liệu lệch khi chọn khóa phân khối lỏng hơn (ví dụ chỉ năm sinh).
3. Tỷ lệ thiếu `diem_qt` cao ở khoa Ngoại ngữ (MAR theo khoa). Điền trung vị toàn cục gây sai lệch gì? Nếu thiếu là MNAR (sinh viên điểm thấp không nộp bài), cách điền nào cũng có thể chệch; đề xuất phân tích độ nhạy.
4. Chỉ số "kịp thời" không đạt ngưỡng do sự kiện từ thiết bị ngoại tuyến. Đây là lỗi dữ liệu hay đặc tính của nguồn? Ngưỡng và SLA nên được thỏa thuận với ai?

---
## Phần 6. ETL vào hồ dữ liệu và mô hình chi phí đám mây

### Khái niệm từ bài giảng
*Slide mục 2.2: "Ba vùng dữ liệu: thô, đã xử lý, phục vụ", "Định dạng dữ liệu và tuần tự hóa", "Apache NiFi" (vùng cách ly); mục 2.3: "Hiệu năng truy cập: thời gian quét song song", "Tầng lưu trữ nóng, ấm, lạnh", "Mô hình chi phí và điểm hòa vốn", "Truyền dữ liệu ra (egress)".*

- **Vùng thô** (raw/bronze): bất biến, giữ định dạng nguồn kèm thời điểm nạp. **Vùng đã xử lý** (silver): lược đồ chuẩn, đã khử trùng lặp, kiểm tra quy tắc, Parquet. **Vùng phục vụ** (gold): bảng tổng hợp, data mart. Bản ghi vi phạm quy tắc đi vào **vùng cách ly** (quarantine) kèm lý do.
- Thời gian quét song song $T \approx D/(n \cdot b)$; định dạng cột và **cắt tỉa phân vùng** (partition pruning) giảm $D$.
- Chi phí đám mây hằng tháng $C = \sum_i S_i p_i + Q p_q + E p_e + H p_h$. Ví dụ trên slide (đơn vị tiền quy ước, đvt, **giả định**): nội bộ $100 + 2t$, đám mây $5t$ ⇒ $t^* \approx 33$ tháng.

### Ví dụ mẫu 6a — Raw → silver cho nhật ký LMS, kèm vùng cách ly và đối soát

In [ ]:
LAKE = f"{DATA_DIR}/lake"
SILVER, GOLD, QUAR = f"{LAKE}/silver", f"{LAKE}/gold", f"{LAKE}/quarantine"

# Vùng thô: sao chép nguyên trạng tệp nguồn, gắn thời điểm nạp trong đường dẫn
shutil.copytree(f"{DATA_DIR}/nguon/lms", f"{RAW}/lms/ngay_nap=2025-09-29", dirs_exist_ok=True)

ds_sv = sv_raw.select("ma_sv")
cach_ly_hong = lms_raw.where(F.col("_corrupt_record").isNotNull()).select(
    F.lit("lms").alias("nguon"), F.lit("JSON hỏng").alias("ly_do"), F.col("_corrupt_record").alias("ban_ghi"))
cach_ly_khach = lms_hop_le.join(ds_sv, "ma_sv", "left_anti").select(
    F.lit("lms").alias("nguon"), F.lit("ma_sv không có trong danh mục sinh viên").alias("ly_do"),
    F.to_json(F.struct(*lms_hop_le.columns)).alias("ban_ghi"))
cach_ly_lms = cach_ly_hong.unionByName(cach_ly_khach)

w_ev = Window.partitionBy("event_id").orderBy("thoi_gian_nap")
lms_silver = (lms_hop_le.join(ds_sv, "ma_sv", "left_semi")
              .withColumn("rn", F.row_number().over(w_ev)).where("rn = 1").drop("rn")
              .withColumn("tre_nap_giay", F.unix_timestamp("thoi_gian_nap") - F.unix_timestamp("thoi_gian"))
              .withColumn("ngay", F.to_date("thoi_gian")))
lms_silver.write.mode("overwrite").partitionBy("ngay").parquet(f"{SILVER}/lms_su_kien")
cach_ly_lms.write.mode("overwrite").parquet(f"{QUAR}/lms")

n_raw, n_sil, n_q = lms_raw.count(), spark.read.parquet(f"{SILVER}/lms_su_kien").count(), spark.read.parquet(f"{QUAR}/lms").count()
n_trung_bo = lms_hop_le.join(ds_sv, "ma_sv", "left_semi").count() - n_sil
print(f"Đối soát: raw {n_raw} = silver {n_sil} + cách ly {n_q} + trùng lặp đã bỏ {n_trung_bo}")
assert n_raw == n_sil + n_q + n_trung_bo
print("Thư mục phân vùng silver:", sorted(os.listdir(f"{SILVER}/lms_su_kien"))[:4], "...")

### Bài tập 6.1 — Quy tắc chất lượng và vùng cách ly cho bảng điểm
Viết `kiem_tra_quy_tac_diem(diem, sv)` trả về `(diem_hop_le, diem_cach_ly)`; `diem_cach_ly` có thêm cột `ly_do` (nối các quy tắc vi phạm bằng `"; "`):
- **R1** điểm ngoài miền $[0, 10]$ (`diem_qt`, `diem_tk` được phép null; `diem_thi` null cũng là vi phạm);
- **R2** `ma_sv` không tồn tại trong danh mục `sv`;
- **R3** $|\text{diem\_tk} - (0{,}4\,\text{diem\_qt} + 0{,}6\,\text{diem\_thi})| > 0{,}06$ khi đủ ba điểm.

Sau đó ghi `diem_hop_le` vào silver (phân vùng theo `hoc_ky`) và `diem_cach_ly` vào vùng cách ly, đều dạng Parquet.

In [ ]:
# BÀI TẬP 6.1 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def kiem_tra_quy_tac_diem(diem, sv):
    ds = sv.select("ma_sv").distinct().withColumn("_co_sv", F.lit(True))
    d = diem.join(ds, "ma_sv", "left")
    # Gợi ý: ngoai = lambda c: (F.col(c) < 0) | (F.col(c) > 10); dùng F.coalesce(..., F.lit(False/True)) cho null
    # r1 = ...; r2 = F.col("_co_sv").isNull(); r3 = ...
    # ly_do = F.concat_ws("; ", F.when(r1, "R1 ..."), F.when(r2, "R2 ..."), F.when(r3, "R3 ..."))  (concat_ws bỏ qua null)
    ...
    return None, None


diem_hop_le, diem_cach_ly = kiem_tra_quy_tac_diem(diem_raw, sv_raw)
# Gợi ý: diem_hop_le.write.mode("overwrite").partitionBy("hoc_ky").parquet(f"{SILVER}/diem")
#        diem_cach_ly.write.mode("overwrite").parquet(f"{QUAR}/diem")

In [ ]:
# Ô TỰ KIỂM TRA
s = SU_THAT_SINH
n_hl, n_cl = diem_hop_le.count(), diem_cach_ly.count()
assert n_hl + n_cl == s["n_diem"], "hợp lệ + cách ly phải bằng tổng số dòng"
assert n_cl == s["n_diem_ngoai_mien"] + s["n_diem_khong_nhat_quan"] + s["n_diem_mo_coi"], f"số dòng cách ly {n_cl} chưa khớp"
for ma, ky_vong in [("R1", s["n_diem_ngoai_mien"]), ("R2", s["n_diem_mo_coi"]), ("R3", s["n_diem_khong_nhat_quan"])]:
    assert diem_cach_ly.where(F.col("ly_do").contains(ma)).count() == ky_vong, ma
assert spark.read.parquet(f"{SILVER}/diem").count() == n_hl and spark.read.parquet(f"{QUAR}/diem").count() == n_cl
assert {"hoc_ky=2024-2", "hoc_ky=2025-1"} <= set(os.listdir(f"{SILVER}/diem")), "silver diem phải phân vùng theo hoc_ky"
dat("Bài tập 6.1")

### Ví dụ mẫu 6b — Vùng phục vụ (gold), cắt tỉa phân vùng và so sánh CSV với Parquet

In [ ]:
silver_lms = spark.read.parquet(f"{SILVER}/lms_su_kien")
gold = (silver_lms.join(sv_raw.select("ma_sv", "ma_khoa"), "ma_sv")
        .groupBy("ngay", "ma_khoa")
        .agg(F.count("*").alias("so_su_kien"), F.approx_count_distinct("ma_sv").alias("sv_hoat_dong"),
             F.round(F.expr("percentile_approx(thoi_luong_giay, 0.5)"), 0).alias("trung_vi_thoi_luong")))
gold.write.mode("overwrite").partitionBy("ngay").parquet(f"{GOLD}/hoat_dong_lms_theo_khoa")

# Cắt tỉa phân vùng: chỉ đọc thư mục ngay=2025-09-20
truy_van = spark.read.parquet(f"{SILVER}/lms_su_kien").where("ngay = '2025-09-20'").select("ma_sv", "thoi_luong_giay")
ke_hoach = truy_van._jdf.queryExecution().executedPlan().toString()
print("PartitionFilters trong kế hoạch thực thi:", re.search(r"PartitionFilters: \[[^\]]*\]", ke_hoach).group(0))

# Kích thước và thời gian truy vấn: CSV, CSV nén gzip, Parquet (snappy)
def kich_thuoc_mb(p):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs if not f.startswith((".", "_"))) / 1e6

silver_lms.write.mode("overwrite").option("header", True).csv(f"{LAKE}/so_sanh/csv")
silver_lms.write.mode("overwrite").option("header", True).option("compression", "gzip").csv(f"{LAKE}/so_sanh/csv_gzip")
silver_lms.write.mode("overwrite").parquet(f"{LAKE}/so_sanh/parquet")
schema_csv = silver_lms.schema
dong = []
for ten, p, doc in [("CSV", f"{LAKE}/so_sanh/csv", lambda p: spark.read.schema(schema_csv).option("header", True).csv(p)),
                    ("CSV gzip", f"{LAKE}/so_sanh/csv_gzip", lambda p: spark.read.schema(schema_csv).option("header", True).csv(p)),
                    ("Parquet", f"{LAKE}/so_sanh/parquet", lambda p: spark.read.parquet(p))]:
    t0 = time.perf_counter()
    doc(p).where("loai = 'lam_quiz'").agg(F.avg("diem_quiz")).first()
    dong.append({"định dạng": ten, "kích thước (MB)": round(kich_thuoc_mb(p), 2),
                 "truy vấn 2 cột (s)": round(time.perf_counter() - t0, 2)})
bang_dd = pd.DataFrame(dong)
print(bang_dd.to_string(index=False))
print("Ở quy mô mặc định, thời gian truy vấn chủ yếu là chi phí khởi tạo tác vụ; lợi thế của Parquet (chỉ đọc cột cần, "
      "đẩy điều kiện lọc xuống) thể hiện rõ khi tăng N_LMS.")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(bang_dd["định dạng"], bang_dd["kích thước (MB)"], color=["#bdbdbd", "#969696", "#2b5d8a"])
ax.set_ylabel("MB"); ax.set_title("Kích thước cùng một bảng silver theo định dạng")
plt.tight_layout(); plt.show()

### Ví dụ mẫu 6c — Chi phí lũy kế và điểm hòa vốn (tham số **giả định**)
Tham số lấy từ slide "Mô hình chi phí và điểm hòa vốn"; đơn vị tiền quy ước (đvt), **không phải giá thực của nhà cung cấp nào**.

In [ ]:
DAU_TU_NOI_BO, VAN_HANH_NOI_BO, DAM_MAY_DAY_TAI = 100, 2, 5     # GIẢ ĐỊNH (đvt, đvt/tháng)
thang = np.arange(0, 61)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(thang, DAU_TU_NOI_BO + VAN_HANH_NOI_BO * thang, color="#8c2d04", lw=2, label="Nội bộ: 100 + 2t")
for u, ls in [(1.0, "-"), (0.6, "--"), (0.4, ":")]:
    ax.plot(thang, DAM_MAY_DAY_TAI * u * thang, color="#2b5d8a", ls=ls, lw=2, label=f"Đám mây, dùng {int(u * 100)}% thời gian: {DAM_MAY_DAY_TAI * u:g}t")
ax.axvline(100 / 3, color="gray", ls="--"); ax.text(34.5, 20, "t* ≈ 33 tháng", color="gray")
ax.set_xlabel("Tháng"); ax.set_ylabel("Chi phí lũy kế (đvt)"); ax.legend(fontsize=10)
ax.set_title("Chi phí lũy kế theo mô hình triển khai (số liệu giả định)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Bài tập 6.2 — Điểm hòa vốn và chi phí lưu trữ theo vòng đời
1. Viết `diem_hoa_von(dau_tu, van_hanh_noi_bo, chi_phi_dam_may)` trả về $t^* = \dfrac{\text{dau\_tu}}{\text{chi\_phi\_dam\_may} - \text{van\_hanh\_noi\_bo}}$ (tháng), hoặc `None` nếu đám mây không đắt hơn vận hành nội bộ.
2. Viết `chi_phi_dam_may_thang(S, p, Q, p_q, E, p_e, H, p_h)` theo công thức $C = \sum_i S_i p_i + Q p_q + E p_e + H p_h$ (`S`, `p` là `dict` theo tầng).
3. Kịch bản **giả định**: mỗi tháng phát sinh 2 TB dữ liệu mới, trong 36 tháng. Chính sách A: mọi dữ liệu ở tầng nóng. Chính sách B (vòng đời): dữ liệu dưới 3 tháng tuổi ở tầng nóng, 3–11 tháng ở tầng ấm, từ 12 tháng ở tầng lạnh. Tính `tong_A`, `tong_B` (tổng chi phí 36 tháng).

In [ ]:
# Tham số GIẢ ĐỊNH (đvt): giá lưu trữ theo TB-tháng, phí yêu cầu, truyền ra, giờ máy
GIA_TANG = {"nong": 1.0, "am": 0.5, "lanh": 0.1}
Q_TRIEU, P_Q = 2, 0.4          # 2 triệu yêu cầu/tháng × 0,4 đvt/triệu
E_TB, P_E = 5, 0.9             # 5 TB truyền ra/tháng × 0,9 đvt/TB
H_GIO, P_H = 300, 0.05         # 300 giờ máy/tháng × 0,05 đvt/giờ
TB_MOI_THANG, SO_THANG = 2, 36

In [ ]:
# BÀI TẬP 6.2 — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
def diem_hoa_von(dau_tu, van_hanh_noi_bo, chi_phi_dam_may):
    # Gợi ý: trả None nếu chi_phi_dam_may <= van_hanh_noi_bo
    ...
    return None


def chi_phi_dam_may_thang(S, p, Q, p_q, E, p_e, H, p_h):
    ...
    return None


def dung_luong_theo_tang(m, vong_doi):
    """Dung lượng (TB) mỗi tầng ở tháng m (1..36); lô dữ liệu sinh ở tháng k có tuổi m - k."""
    S = {"nong": 0.0, "am": 0.0, "lanh": 0.0}
    # Gợi ý: với k = 1..m, tuoi = m - k; chọn tầng theo chính sách; cộng TB_MOI_THANG vào tầng đó
    ...
    return S


tong_A = ...   # Gợi ý: tổng chi_phi_dam_may_thang(dung_luong_theo_tang(m, False), GIA_TANG, ...) với m = 1..SO_THANG
tong_B = ...

In [ ]:
# Ô TỰ KIỂM TRA
assert abs(diem_hoa_von(100, 2, 5) - 100 / 3) < 1e-9, "ví dụ trên slide: t* ≈ 33 tháng"
assert diem_hoa_von(100, 2, 2) is None, "đám mây 2 đvt/tháng (tắt cụm khi rảnh): không có điểm hòa vốn"
assert abs(chi_phi_dam_may_thang({"nong": 10, "lanh": 100}, {"nong": 1.0, "lanh": 0.1}, 2, 0.4, 5, 0.9, 300, 0.05) - 40.3) < 1e-9
co_dinh = SO_THANG * (Q_TRIEU * P_Q + E_TB * P_E + H_GIO * P_H)
assert abs(tong_A - (sum(2 * m for m in range(1, 37)) * 1.0 + co_dinh)) < 1e-6
ky_vong_B = co_dinh + sum(2 * (1.0 * min(m, 3) + 0.5 * max(0, min(m, 12) - 3) + 0.1 * max(0, m - 12)) for m in range(1, 37))
assert abs(tong_B - ky_vong_B) < 1e-6
# Độ nhạy: tỷ lệ thời gian sử dụng u của cụm đám mây → điểm hòa vốn
u = np.linspace(0.3, 1.0, 71)
t_sao = [diem_hoa_von(100, 2, 5 * x) for x in u]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot([x for x, t in zip(u, t_sao) if t is not None], [t for t in t_sao if t is not None], color="#2b5d8a", lw=2)
ax.axhline(60, color="gray", ls="--", label="vòng đời thiết bị 60 tháng (giả định)")
ax.set_ylim(0, 150); ax.set_xlabel("Tỷ lệ thời gian sử dụng cụm đám mây u"); ax.set_ylabel("Điểm hòa vốn t* (tháng)")
ax.set_title("Độ nhạy của điểm hòa vốn theo mức sử dụng (giả định)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Khi u < {(100 / 60 + 2) / 5:.2f}, điểm hòa vốn vượt 60 tháng → đám mây có lợi trong suốt vòng đời thiết bị (theo giả định).")
dat("Bài tập 6.2")

### Câu hỏi thảo luận — Phần 6
1. (Câu hỏi thảo luận 3 của chương) Trong mô hình chi phí, tham số nào làm dịch chuyển điểm hòa vốn mạnh nhất? Khi nào mô hình **lai** (dữ liệu cá nhân ở nội bộ, tính toán co giãn trên đám mây cho dữ liệu ẩn danh) có lợi hơn cả hai phương án thuần? Bổ sung chi phí truyền ra (egress) vào mô hình.
2. Từ Ví dụ 6b, ước lượng thời gian quét một năm nhật ký LMS (giả định 50 TB CSV) trên cụm 20 nút, 200 MB/s mỗi nút, trước và sau khi chuyển sang Parquet và chỉ đọc 3/10 cột, 1/12 phân vùng.
3. Vùng thô được giữ bất biến nhưng chứa dữ liệu cá nhân (số điện thoại, ngày sinh). Thiết kế chính sách lưu giữ, phân quyền và ẩn danh hóa phù hợp với khung pháp lý về bảo vệ dữ liệu cá nhân.

---
## Bài tập mở rộng / Mini-project — Pipeline thu thập cho nguồn dữ liệu thư viện

*Liên hệ slide: "Pipeline thu thập cho trung tâm dữ liệu đại học", "Tình huống: quyết định thiết kế và đánh đổi".*

Thư viện trường dùng một phần mềm riêng, mỗi ngày **xuất tệp CSV** giao dịch mượn–trả (dấu phân cách `;`), kèm danh mục bạn đọc. Khoảng 30% thẻ bạn đọc được cấp trước khi liên thông nên **không có mã sinh viên**. Phần mềm đôi khi xuất trùng dòng và có ngày phải **xuất lại** tệp có sửa hạn trả (`..._v2.csv`).

**Yêu cầu** (làm theo nhóm 2–3 học viên, nộp notebook và báo cáo 2 trang):
1. **Chọn phương thức thu thập** (kéo theo lô, đẩy, luồng, CDC) và công cụ trên cụm thật (ví dụ NiFi `GetFile`/`ListSFTP`, Spark batch); lập luận bằng bảng tiêu chí của slide (độ trễ yêu cầu, tải nguồn, khả năng bắt sửa/xóa, độ phức tạp vận hành).
2. **Nạp vùng thô** với lược đồ khai báo; giữ nguyên tệp nguồn và thời điểm nạp.
3. **Làm sạch và tích hợp:** chuẩn hóa họ tên/ngày/số điện thoại (tái sử dụng hàm Phần 5); khử trùng lặp và giữ **phiên bản mới nhất** của giao dịch khi tệp được xuất lại; liên kết bạn đọc với sinh viên theo `ma_sv` hoặc bằng **đối sánh thực thể**.
4. **Bảng điểm chất lượng** tối thiểu năm chiều, có ngưỡng; bản ghi vi phạm vào **vùng cách ly** kèm lý do.
5. **Ghi silver/gold** dạng Parquet phân vùng; bảng gold gợi ý: số lượt mượn theo khoa và ngày, tỷ lệ trả trễ theo khoa.
6. **Ước lượng chi phí** lưu trữ 5 năm cho nguồn này theo mô hình Phần 6 (tham số giả định, ghi rõ).

| Tiêu chí đánh giá | Trọng số |
|---|---|
| Lý do chọn phương thức và công cụ thu thập, có số liệu | 20% |
| Pipeline chạy được, lũy đẳng khi chạy lại cùng một ngày | 25% |
| Làm sạch, khử trùng lặp, đối sánh thực thể (có độ đo) | 25% |
| Bảng điểm chất lượng và vùng cách ly | 20% |
| Ước lượng chi phí và thảo luận đánh đổi | 10% |

In [ ]:
# Sinh dữ liệu mô phỏng cho nguồn thư viện (chạy được ở cả hai bản notebook)
def sinh_du_lieu_thu_vien(seed=SEED + 7, so_ngay=7):
    r = random.Random(seed)
    thu_muc = f"{DATA_DIR}/nguon/thu_vien"
    os.makedirs(thu_muc, exist_ok=True)
    ban_doc = []
    for k, g in enumerate(r.sample(sv_goc, 1500)):
        ban_doc.append({"ma_the": f"TV{k + 1:05d}", "ma_sv": g["ma_sv"] if r.random() < 0.7 else "",
                        "ho_ten": lam_ban_ten(g["ho_ten"], r), "ngay_sinh": lam_ban_ngay(g["ngay_sinh"], r) or "",
                        "so_dt": lam_ban_sdt(g["so_dt"], r) or ""})
    pd.DataFrame(ban_doc).to_csv(f"{thu_muc}/ban_doc.csv", index=False, sep=";")
    gd_id = 0
    for d in range(so_ngay):
        ngay = date(2025, 9, 15) + timedelta(days=d)
        dong = []
        for _ in range(r.randint(500, 700)):
            gd_id += 1
            t = datetime.combine(ngay, datetime.min.time()) + timedelta(seconds=r.randint(7 * 3600, 20 * 3600))
            dong.append({"ma_gd": f"GD{gd_id:07d}", "ma_the": r.choice(ban_doc)["ma_the"] if r.random() > 0.005 else "TV99999",
                         "ma_sach": f"S{r.randint(1, 5000):05d}", "loai": r.choices(["muon", "tra", "gia_han"], [0.5, 0.4, 0.1])[0],
                         "thoi_diem": t.strftime("%d/%m/%Y %H:%M"), "han_tra": (ngay + timedelta(days=14)).strftime("%d/%m/%Y")})
        dong += r.sample(dong, k=len(dong) // 100)                                  # xuất trùng ~1%
        pd.DataFrame(dong).to_csv(f"{thu_muc}/muon_tra_{ngay.isoformat()}.csv", index=False, sep=";")
        if d == 2:                                                                  # ngày 17/09 xuất lại có sửa hạn trả
            sua = pd.DataFrame(dong).drop_duplicates("ma_gd")
            sua.loc[sua.sample(n=20, random_state=seed).index, "han_tra"] = (ngay + timedelta(days=21)).strftime("%d/%m/%Y")
            sua.to_csv(f"{thu_muc}/muon_tra_{ngay.isoformat()}_v2.csv", index=False, sep=";")
    return thu_muc


THU_MUC_TV = sinh_du_lieu_thu_vien()
print(sorted(os.listdir(THU_MUC_TV)))
print(open(f"{THU_MUC_TV}/muon_tra_2025-09-15.csv", encoding="utf-8").read()[:300])

In [ ]:
# BÀI TẬP mini-project (lời giải tham khảo rút gọn) — điền vào các chỗ `...` / None rồi chạy ô kiểm tra bên dưới
# Khung gợi ý — tự thiết kế và mở rộng; trình bày lý do chọn phương thức thu thập trong ô markdown riêng.
# 1) Vùng thô: lược đồ khai báo cho ban_doc.csv và muon_tra_*.csv (sep=";"); giữ F.input_file_name() làm phả hệ
bd_raw = ...
mt_raw = ...
# 2) Làm sạch: chuẩn hóa thời gian (dd/MM/yyyy HH:mm), giữ phiên bản mới nhất theo ma_gd (tệp _v2 ưu tiên)
# 3) Liên kết bạn đọc ↔ sinh viên: theo ma_sv nếu có; nếu không, dùng chuan_hoa_sinh_vien + doi_sanh_thuc_the (Phần 5)
# 4) Bảng điểm chất lượng ≥ 5 chiều + vùng cách ly (f"{QUAR}/thu_vien_...")
# 5) Ghi silver/gold Parquet phân vùng theo ngày (f"{SILVER}/...", f"{GOLD}/...")
# 6) Ước lượng chi phí lưu trữ 5 năm (dùng chi_phi_dam_may_thang ở Phần 6, tham số giả định)

---
## Dọn dẹp và tài liệu tham khảo

In [ ]:
spark.stop()
XOA_DU_LIEU = False    # đặt True để xóa ./data_c2 và ./spark-warehouse-c2 sau khi hoàn thành
if XOA_DU_LIEU:
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    shutil.rmtree("spark-warehouse-c2", ignore_errors=True)
print("Đã dừng SparkSession.", "Đã xóa dữ liệu tạm." if XOA_DU_LIEU else f"Dữ liệu vẫn ở {DATA_DIR}.")

**Tài liệu tham khảo**
1. A. Bahga, V. Madisetti. *Big Data Analytics: A Hands-On Approach*. VPT, 2019 — chương về thu thập và tích hợp dữ liệu.
2. T. White. *Hadoop: The Definitive Guide*, 4th ed. O'Reilly, 2015 — Sqoop, Flume.
3. J. Kreps, N. Narkhede, J. Rao. Kafka: a distributed messaging system for log processing. *NetDB*, 2011.
4. M. Kleppmann. *Designing Data-Intensive Applications*. O'Reilly, 2017 — chương 11 (luồng sự kiện, CDC).
5. M. Greenwald, S. Khanna. Space-efficient online computation of quantile summaries. *SIGMOD*, 2001.
6. S. Heule, M. Nunkesser, A. Hall. HyperLogLog in practice. *EDBT*, 2013.
7. I. P. Fellegi, A. B. Sunter. A theory for record linkage. *JASA*, 64(328), 1969.
8. D. B. Rubin. Inference and missing data. *Biometrika*, 63(3), 1976.
9. R. Y. Wang, D. M. Strong. Beyond accuracy: what data quality means to data consumers. *JMIS*, 12(4), 1996.
10. S. Schelter et al. Automating large-scale data quality verification (Deequ). *PVLDB*, 11(12), 2018.
11. P. Mell, T. Grance. *The NIST Definition of Cloud Computing*. NIST SP 800-145, 2011.
12. Lê Việt Thuỷ, Nguyễn Văn Hoàng. Xây dựng trung tâm dữ liệu lớn cho hệ thống đại học thông minh. Hội thảo khoa học quốc gia "Ứng dụng CNTT trong đổi mới phương pháp giảng dạy theo hướng Blended Learning", tr. 209–218.
13. Tài liệu Apache Spark 3.5: *JDBC To Other Databases*, *JSON Files*, `DataFrameStatFunctions.approxQuantile`, `pyspark.ml.feature.Imputer`.